Multi-Agent System for Hospital Appointment Scheduling

Introduction

Hospital appointment scheduling is a complex operational task that requires coordinating patients, physicians, medical specialties, available time slots, patient preferences, and scheduling constraints. In real healthcare environments, scheduling is also dynamic because appointments may be cancelled, new requests may arrive, physicians may become temporarily unavailable, and doctors may change their schedules.

This project investigates whether a Multi-Agent System (MAS) can improve hospital appointment scheduling compared with a Single-Agent baseline under the same information, constraints, and scheduling environment.

The project follows a controlled experimental approach. First, a Single-Agent scheduling system is developed as the baseline. The baseline centrally handles appointment requests by identifying feasible appointments, evaluating candidate assignments according to patient preferences, and selecting the best available option. The resulting schedules are then validated against the defined constraints and evaluated using several performance metrics.

The experiments are conducted using synthetic hospital data, including physicians, patients, appointment requests, and time slots. The first experiment uses a relatively simple dataset to establish the basic scheduling behaviour of the baseline system. The second experiment increases the problem complexity by introducing a larger dataset, multiple appointment requests per patient, different appointment durations, physician capacity constraints, maximum waiting periods, and dynamic events.

The dynamic experiment evaluates how the Single-Agent system responds when the scheduling environment changes after the initial schedule has been created. These changes include patient cancellations, physician unavailability, new appointment requests, and physician schedule changes.

The same datasets, constraints, events, and evaluation metrics will later be used to evaluate the proposed Multi-Agent System. This controlled setup allows the performance of the Single-Agent and Multi-Agent approaches to be compared under equivalent conditions.

The main research question addressed by this project is:

Does a Multi-Agent System improve hospital appointment scheduling performance compared with a Single-Agent approach as the scheduling environment becomes more complex and dynamic?

The main evaluation criteria include:

• Successful assignment rate
• Constraint violations
• Patient preference satisfaction
• Date and time satisfaction
• Average waiting or date deviation
• Dynamic event success rate
• Appointment recovery after disruptions

The results of the Single-Agent experiments therefore provide the baseline performance against which the Multi-Agent System will be compared.

Experiment 1 — Easy Dataset: Single-Agent Baseline

This experiment establishes the initial Single-Agent scheduling baseline using a controlled and relatively simple synthetic hospital dataset.

The purpose of this experiment is to verify the basic scheduling workflow before introducing larger datasets and dynamic changes. The system generates physicians, patients, appointment requests, and available time slots, and then uses a centralized scheduling agent to assign suitable appointments.

The resulting schedule is validated against the defined scheduling constraints and evaluated using patient satisfaction and assignment performance metrics.

Doctors Dataset

This section creates a synthetic dataset representing physicians in a simplified hospital environment.

Each physician is assigned a medical specialty, years of experience, working hours, and a maximum number of appointments that can be handled per day. These attributes represent the basic resources and constraints required for appointment scheduling.

The physician dataset will later be shared between the Single-Agent and Multi-Agent systems so that both approaches operate under the same healthcare environment and can be compared fairly.

In [1]:
import pandas as pd
import random

random.seed(42)

SPECIALTIES = [
    "Cardiology",
    "Dermatology",
    "Neurology",
    "Pediatrics",
    "Orthopedics",
    "General Medicine",
    "Internal Medicine",
    "ENT",
    "Ophthalmology",
    "Obstetrics and Gynecology"
]

def generate_doctors(num_doctors):
    doctors = []

    for i in range(1, num_doctors + 1):
        specialty = SPECIALTIES[(i - 1) % len(SPECIALTIES)]

        doctors.append({
            "doctor_id": f"D{i:03d}",
            "specialty": specialty,
            "experience_years": random.randint(3, 20),
            "work_start": "09:00",
            "work_end": "17:00",
            "max_daily_appointments": random.randint(1, 8)
        })

    return pd.DataFrame(doctors)

doctors_df = generate_doctors(20)

print("===== DOCTORS DATASET =====")
print("Number of doctors:", len(doctors_df))
print()
print(doctors_df)

===== DOCTORS DATASET =====
Number of doctors: 20

   doctor_id                  specialty  ...  work_end max_daily_appointments
0       D001                 Cardiology  ...     17:00                      1
1       D002                Dermatology  ...     17:00                      4
2       D003                  Neurology  ...     17:00                      3
3       D004                 Pediatrics  ...     17:00                      2
4       D005                Orthopedics  ...     17:00                      1
5       D006           General Medicine  ...     17:00                      2
6       D007          Internal Medicine  ...     17:00                      4
7       D008                        ENT  ...     17:00                      1
8       D009              Ophthalmology  ...     17:00                      4
9       D010  Obstetrics and Gynecology  ...     17:00                      7
10      D011                 Cardiology  ...     17:00                      8
11      D012 

Doctors Dataset Result

The generated dataset contains 20 physicians distributed across 10 medical specialties. Each physician has defined working hours and a daily appointment capacity.

This dataset provides the available physician resources that the scheduling system can use when matching patients with appropriate doctors.

The use of synthetic but structured physician data allows the scheduling process to be controlled while maintaining the main characteristics required for a hospital appointment scheduling problem.

Patients Dataset

This section generates the synthetic patient dataset used by the scheduling system.

Each patient is assigned a required medical specialty and may have a preferred physician. Patients also specify a preferred appointment date and time period. These preferences are important because the scheduling system should not only produce valid appointments, but should also attempt to satisfy patient preferences whenever possible.

The same patient dataset will be used by both scheduling approaches to maintain a fair experimental comparison.

In [2]:
# Part 2 — Patients Dataset

from datetime import datetime, timedelta

PATIENT_COUNT = 100

PREFERRED_TIME_OPTIONS = [
    "Morning",
    "Afternoon",
    "No Preference"
]

def generate_patients(num_patients, doctors_df, start_date):
    patients = []

    for i in range(1, num_patients + 1):
        patient_id = f"P{i:03d}"

        required_specialty = random.choice(SPECIALTIES)

        specialty_doctors = doctors_df[
            doctors_df["specialty"] == required_specialty
        ]["doctor_id"].tolist()

        if random.random() < 0.40:
            preferred_doctor_id = random.choice(specialty_doctors)
        else:
            preferred_doctor_id = None

        preferred_date = start_date + timedelta(
            days=random.randint(0, 6)
        )

        preferred_time = random.choice(PREFERRED_TIME_OPTIONS)

        patients.append({
            "patient_id": patient_id,
            "required_specialty": required_specialty,
            "preferred_doctor_id": preferred_doctor_id,
            "preferred_date": preferred_date.strftime("%Y-%m-%d"),
            "preferred_time": preferred_time
        })

    return pd.DataFrame(patients)


START_DATE = datetime(2026, 9, 14)

patients_df = generate_patients(
    PATIENT_COUNT,
    doctors_df,
    START_DATE
)

print("===== PATIENTS DATASET =====")
print("Number of patients:", len(patients_df))
print()

print(patients_df.head(10))

===== PATIENTS DATASET =====
Number of patients: 100

  patient_id         required_specialty  ... preferred_date preferred_time
0       P001              Ophthalmology  ...     2026-09-14  No Preference
1       P002                Orthopedics  ...     2026-09-18      Afternoon
2       P003  Obstetrics and Gynecology  ...     2026-09-14  No Preference
3       P004                 Pediatrics  ...     2026-09-14        Morning
4       P005                Dermatology  ...     2026-09-19      Afternoon
5       P006                  Neurology  ...     2026-09-19      Afternoon
6       P007                Dermatology  ...     2026-09-15  No Preference
7       P008                 Pediatrics  ...     2026-09-16  No Preference
8       P009              Ophthalmology  ...     2026-09-20        Morning
9       P010                 Pediatrics  ...     2026-09-20      Afternoon

[10 rows x 5 columns]


Patients Dataset Result

The generated dataset contains 100 patients. Each patient has a required medical specialty and scheduling preferences, including a preferred date and time. Some patients also specify a preferred physician, while others allow the system to select any suitable physician.

This dataset provides the patient-side requirements that the scheduling algorithm must satisfy while considering physician availability and scheduling constraints.

Appointment Requests

This section converts the patient information into standardized appointment requests.

Each request represents a patient’s need to schedule an appointment and contains the information required by the scheduling system, including the patient identifier, required specialty, preferred physician, preferred date, and preferred time.

Using standardized appointment requests provides a common input format that can later be supplied to both the Single-Agent and Multi-Agent systems.

In [3]:
# Part 3 — Appointment Requests

def generate_appointment_requests(patients_df):
    requests = []

    for _, patient in patients_df.iterrows():

        request_id = f"R{len(requests) + 1:03d}"

        requests.append({
            "request_id": request_id,
            "patient_id": patient["patient_id"],
            "required_specialty": patient["required_specialty"],
            "preferred_doctor_id": patient["preferred_doctor_id"],
            "preferred_date": patient["preferred_date"],
            "preferred_time": patient["preferred_time"]
        })

    return pd.DataFrame(requests)


appointment_requests_df = generate_appointment_requests(
    patients_df
)

print("===== APPOINTMENT REQUESTS =====")
print("Number of appointment requests:", len(appointment_requests_df))
print()

print(appointment_requests_df.head(10))

===== APPOINTMENT REQUESTS =====
Number of appointment requests: 100

  request_id patient_id  ... preferred_date preferred_time
0       R001       P001  ...     2026-09-14  No Preference
1       R002       P002  ...     2026-09-18      Afternoon
2       R003       P003  ...     2026-09-14  No Preference
3       R004       P004  ...     2026-09-14        Morning
4       R005       P005  ...     2026-09-19      Afternoon
5       R006       P006  ...     2026-09-19      Afternoon
6       R007       P007  ...     2026-09-15  No Preference
7       R008       P008  ...     2026-09-16  No Preference
8       R009       P009  ...     2026-09-20        Morning
9       R010       P010  ...     2026-09-20      Afternoon

[10 rows x 6 columns]


Appointment Requests Result

A total of 100 appointment requests were generated, corresponding to the 100 patients in the Easy experiment.

Each request provides the scheduling requirements that the Single-Agent scheduler must evaluate. This ensures that the scheduler works with explicit and structured inputs rather than directly modifying the patient dataset.

Time Slots and Doctor Availability

This section creates the time environment used for appointment scheduling.

Each physician’s working day is divided into fixed one-hour time slots over the scheduling period. Every slot contains the physician, date, start time, end time, and current availability status.

These slots represent the hospital’s available scheduling resources. The scheduler can only assign an appointment when a suitable slot is available and satisfies the patient’s and physician’s constraints

In [4]:
# Part 4 — Time Slots & Doctor Availability

WORK_START_TIME = 9
WORK_END_TIME = 17

SLOT_DURATION_MINUTES = 60

SCHEDULING_DAYS = 6


def generate_time_slots(doctors_df, start_date):
    slots = []

    for _, doctor in doctors_df.iterrows():

        doctor_id = doctor["doctor_id"]

        for day_offset in range(SCHEDULING_DAYS):

            current_date = start_date + timedelta(days=day_offset)

            current_time = datetime.combine(
                current_date,
                datetime.min.time()
            ).replace(hour=WORK_START_TIME)

            end_time = datetime.combine(
                current_date,
                datetime.min.time()
            ).replace(hour=WORK_END_TIME)

            while current_time < end_time:

                slot_end_time = current_time + timedelta(
                    minutes=SLOT_DURATION_MINUTES
                )

                slots.append({
                    "slot_id": f"S{len(slots) + 1:04d}",
                    "doctor_id": doctor_id,
                    "date": current_date.strftime("%Y-%m-%d"),
                    "start_time": current_time.strftime("%H:%M"),
                    "end_time": slot_end_time.strftime("%H:%M"),
                    "status": "Available"
                })

                current_time = slot_end_time

    return pd.DataFrame(slots)


slots_df = generate_time_slots(
    doctors_df,
    START_DATE
)


print("===== TIME SLOTS DATASET =====")
print("Number of doctors:", len(doctors_df))
print("Scheduling days:", SCHEDULING_DAYS)
print("Slot duration:", SLOT_DURATION_MINUTES, "minutes")
print("Working hours:", f"{WORK_START_TIME}:00 - {WORK_END_TIME}:00")
print("Total slots:", len(slots_df))
print()

print(slots_df.head(10))

===== TIME SLOTS DATASET =====
Number of doctors: 20
Scheduling days: 6
Slot duration: 60 minutes
Working hours: 9:00 - 17:00
Total slots: 960

  slot_id doctor_id        date start_time end_time     status
0   S0001      D001  2026-09-14      09:00    10:00  Available
1   S0002      D001  2026-09-14      10:00    11:00  Available
2   S0003      D001  2026-09-14      11:00    12:00  Available
3   S0004      D001  2026-09-14      12:00    13:00  Available
4   S0005      D001  2026-09-14      13:00    14:00  Available
5   S0006      D001  2026-09-14      14:00    15:00  Available
6   S0007      D001  2026-09-14      15:00    16:00  Available
7   S0008      D001  2026-09-14      16:00    17:00  Available
8   S0009      D001  2026-09-15      09:00    10:00  Available
9   S0010      D001  2026-09-15      10:00    11:00  Available


Time Slots Result

The experiment contains 20 physicians over 6 scheduling days, with one-hour appointment slots between 09:00 and 17:00.

This produces a total of 960 time slots. All slots are initially available and can be selected by the scheduling system when they satisfy the required constraints.

The generated slots therefore form the basic temporal resource used by the Single-Agent scheduler.

Single-Agent Scheduling Logic

This section implements the core scheduling logic of the Single-Agent baseline.

The Single-Agent centrally performs the main scheduling tasks. For each appointment request, it identifies feasible physician-slot combinations, removes assignments that violate constraints, calculates a score for the remaining candidates, and selects the highest-scoring valid assignment.

The scoring mechanism considers patient preferences such as preferred physician, preferred date, and preferred time. Importantly, feasibility is checked before scoring so that an invalid assignment cannot be selected simply because it receives a high preference score.

This centralized workflow provides the baseline architecture that will later be compared with the Multi-Agent approach.

In [5]:
# Part 5 — Single-Agent Scheduling Logic

from datetime import datetime


# ---------------------------------------------------------
# 1. Check whether the selected time matches the preference
# ---------------------------------------------------------

def matches_time_preference(start_time, preferred_time):

    hour = int(start_time.split(":")[0])

    if preferred_time == "Morning":
        return hour < 12

    elif preferred_time == "Afternoon":
        return hour >= 12

    else:
        return True


# ---------------------------------------------------------
# 2. Calculate the score of a possible assignment
# ---------------------------------------------------------

def calculate_assignment_score(
    patient,
    doctor,
    slot,
    current_date
):

    score = 0

    # Preferred doctor
    if (
        patient["preferred_doctor_id"] is not None
        and patient["preferred_doctor_id"] == doctor["doctor_id"]
    ):
        score += 40

    # Preferred date
    if slot["date"] == patient["preferred_date"]:
        score += 30

    # Preferred time
    if matches_time_preference(
        slot["start_time"],
        patient["preferred_time"]
    ):
        score += 20

    # Earlier appointment
    slot_date = datetime.strptime(
        slot["date"],
        "%Y-%m-%d"
    ).date()

    preference_date = datetime.strptime(
        patient["preferred_date"],
        "%Y-%m-%d"
    ).date()

    days_from_preference = abs(
        (slot_date - preference_date).days
    )

    # Maximum 10 points for the closest date
    score += max(
        0,
        10 - days_from_preference
    )

    return score


# ---------------------------------------------------------
# 3. Check whether an assignment satisfies the constraints
# ---------------------------------------------------------

def is_valid_assignment(
    patient,
    doctor,
    slot,
    schedule,
    doctor_daily_counts
):

    # Specialty compatibility
    if doctor["specialty"] != patient["required_specialty"]:
        return False

    # Slot must be available
    if slot["status"] != "Available":
        return False

    # Doctor daily capacity
    doctor_id = doctor["doctor_id"]
    slot_date = slot["date"]

    current_daily_count = doctor_daily_counts.get(
        (doctor_id, slot_date),
        0
    )

    if current_daily_count >= doctor["max_daily_appointments"]:
        return False

    # Patient cannot have another appointment
    # on the same date and time
    for existing in schedule:

        if (
            existing["patient_id"] == patient["patient_id"]
            and existing["date"] == slot_date
            and existing["start_time"] == slot["start_time"]
        ):
            return False

    return True

Scheduling Logic Result

The scheduling logic successfully defines the main decision process of the Single-Agent baseline.

The process separates feasibility from preference optimization: an assignment must first satisfy the scheduling constraints, and only then is it evaluated according to the patient’s preferences.

This makes the scheduling process interpretable and ensures that preference satisfaction does not override fundamental hospital constraints.

Run the Single-Agent Baseline

This section executes the Single-Agent scheduling process for all appointment requests.

Requests are processed sequentially. For each request, the scheduler searches for feasible appointments, evaluates the available candidates, selects the highest-scoring valid assignment, and reserves the corresponding time slot.

If no feasible appointment can be found, the request is recorded as unscheduled.

In [6]:
# Part 6 — Run Baseline

def run_baseline(
    appointment_requests_df,
    patients_df,
    doctors_df,
    slots_df
):

    # Create working copies
    working_slots = slots_df.copy()

    schedule = []

    doctor_daily_counts = {}

    unscheduled_requests = []

    # -----------------------------------------------------
    # Process each appointment request
    # -----------------------------------------------------

    for _, request in appointment_requests_df.iterrows():

        patient = patients_df[
            patients_df["patient_id"] == request["patient_id"]
        ].iloc[0]

        # -------------------------------------------------
        # Generate candidate assignments
        # -------------------------------------------------

        candidates = []

        for _, doctor in doctors_df.iterrows():

            # Quick specialty filtering
            if doctor["specialty"] != patient["required_specialty"]:
                continue

            doctor_slots = working_slots[
                working_slots["doctor_id"] == doctor["doctor_id"]
            ]

            for slot_index, slot in doctor_slots.iterrows():

                # Check constraints
                if not is_valid_assignment(
                    patient,
                    doctor,
                    slot,
                    schedule,
                    doctor_daily_counts
                ):
                    continue

                # Calculate score
                score = calculate_assignment_score(
                    patient,
                    doctor,
                    slot,
                    START_DATE
                )

                candidates.append({
                    "doctor_id": doctor["doctor_id"],
                    "slot_id": slot["slot_id"],
                    "date": slot["date"],
                    "start_time": slot["start_time"],
                    "end_time": slot["end_time"],
                    "score": score
                })

        # -------------------------------------------------
        # Select the best candidate
        # -------------------------------------------------

        if len(candidates) == 0:

            unscheduled_requests.append({
                "request_id": request["request_id"],
                "patient_id": request["patient_id"],
                "reason": "No valid assignment available"
            })

            continue

        # Sort candidates
        candidates = sorted(
            candidates,
            key=lambda x: (
                -x["score"],
                x["date"],
                x["start_time"]
            )
        )

        best_assignment = candidates[0]

        # -------------------------------------------------
        # Book the selected slot
        # -------------------------------------------------

        selected_slot_id = best_assignment["slot_id"]

        working_slots.loc[
            working_slots["slot_id"] == selected_slot_id,
            "status"
        ] = "Booked"

        # Update doctor's daily appointment count
        doctor_id = best_assignment["doctor_id"]
        appointment_date = best_assignment["date"]

        key = (doctor_id, appointment_date)

        doctor_daily_counts[key] = (
            doctor_daily_counts.get(key, 0) + 1
        )

        # -------------------------------------------------
        # Add assignment to final schedule
        # -------------------------------------------------

        schedule.append({
            "request_id": request["request_id"],
            "patient_id": patient["patient_id"],
            "doctor_id": best_assignment["doctor_id"],
            "required_specialty": patient["required_specialty"],
            "slot_id": best_assignment["slot_id"],
            "date": best_assignment["date"],
            "start_time": best_assignment["start_time"],
            "end_time": best_assignment["end_time"],
            "score": best_assignment["score"]
        })

    return (
        pd.DataFrame(schedule),
        pd.DataFrame(unscheduled_requests),working_slots
    )


# ---------------------------------------------------------
# Run the Baseline
# ---------------------------------------------------------

baseline_schedule_df, unscheduled_df, baseline_slots_df = run_baseline(
    appointment_requests_df,
    patients_df,
    doctors_df,
    slots_df
)


print("===== BASELINE RUN =====")
print("Total requests:", len(appointment_requests_df))
print("Scheduled requests:", len(baseline_schedule_df))
print("Unscheduled requests:", len(unscheduled_df))
print()

print("===== BASELINE SCHEDULE PREVIEW =====")
print(baseline_schedule_df.head(10))

===== BASELINE RUN =====
Total requests: 100
Scheduled requests: 100
Unscheduled requests: 0

===== BASELINE SCHEDULE PREVIEW =====
  request_id patient_id doctor_id  ... start_time end_time score
0       R001       P001      D019  ...      09:00    10:00   100
1       R002       P002      D005  ...      12:00    13:00    60
2       R003       P003      D010  ...      09:00    10:00   100
3       R004       P004      D004  ...      09:00    10:00    60
4       R005       P005      D012  ...      12:00    13:00   100
5       R006       P006      D003  ...      12:00    13:00   100
6       R007       P007      D002  ...      09:00    10:00    60
7       R008       P008      D014  ...      09:00    10:00   100
8       R009       P009      D019  ...      09:00    10:00    69
9       R010       P010      D004  ...      12:00    13:00    29

[10 rows x 9 columns]


Baseline Scheduling Result

The Single-Agent successfully scheduled all 100 appointment requests, with 0 unscheduled requests.

This represents a 100% successful assignment rate for the Easy experiment.

The result indicates that the available physician capacity and time slots were sufficient to satisfy all requests under the constraints used in this controlled experiment.

Final Baseline Schedule

This section organizes and displays the appointments generated by the Single-Agent scheduler.

The schedule is sorted by date, start time, and physician so that the assignments can be inspected in chronological order. The resulting table contains the patient, physician, specialty, appointment date and time, and the calculated assignment score.

Displaying the final schedule also allows the validity of individual assignments to be examined before the formal validation stage.

In [7]:
# Part 7 — Baseline Schedule

baseline_schedule_df = baseline_schedule_df.sort_values(
    by=["date", "start_time", "doctor_id"]
).reset_index(drop=True)


print("===== FINAL BASELINE SCHEDULE =====")
print("Total scheduled appointments:", len(baseline_schedule_df))
print()

print(
    baseline_schedule_df[
        [
            "request_id",
            "patient_id",
            "doctor_id",
            "slot_id",
            "required_specialty",
            "date",
            "start_time",
            "end_time",
            "score"
        ]
    ].to_string(index=False)
)

===== FINAL BASELINE SCHEDULE =====
Total scheduled appointments: 100

request_id patient_id doctor_id slot_id        required_specialty       date start_time end_time  score
      R040       P040      D001   S0001                Cardiology 2026-09-14      09:00    10:00     60
      R050       P050      D002   S0049               Dermatology 2026-09-14      09:00    10:00    100
      R004       P004      D004   S0145                Pediatrics 2026-09-14      09:00    10:00     60
      R078       P078      D007   S0289         Internal Medicine 2026-09-14      09:00    10:00     60
      R021       P021      D009   S0385             Ophthalmology 2026-09-14      09:00    10:00     60
      R003       P003      D010   S0433 Obstetrics and Gynecology 2026-09-14      09:00    10:00    100
      R096       P096      D011   S0481                Cardiology 2026-09-14      09:00    10:00     60
      R001       P001      D019   S0865             Ophthalmology 2026-09-14      09:00    10:00 

Final Schedule Result

The Single-Agent generated 100 scheduled appointments.

The schedule contains valid physician assignments and corresponding appointment times. The assignment score provides an indication of how well each selected appointment matches the patient’s preferences.

The schedule is now ready for formal constraint validation.

Baseline Schedule Validation

This section validates the generated schedule against the predefined scheduling constraints.

Validation is necessary because successfully generating a schedule does not automatically guarantee that all assignments are valid.

The validation process checks important constraints such as physician-specialty compatibility, slot availability, physician capacity, and scheduling conflicts.

Any detected violation is recorded so that the final schedule can be evaluated objectively.

In [8]:
# Part 8 — Baseline Validation

def validate_baseline_schedule(
    baseline_schedule_df,
    doctors_df,
    slots_df
):

    violations = []

    # ---------------------------------------------------------
    # 1. Check specialty compatibility
    # ---------------------------------------------------------

    for _, appointment in baseline_schedule_df.iterrows():

        doctor = doctors_df[
            doctors_df["doctor_id"] == appointment["doctor_id"]
        ].iloc[0]

        if doctor["specialty"] != appointment["required_specialty"]:

            violations.append({
                "patient_id": appointment["patient_id"],
                "violation": "Specialty mismatch"
            })

    # ---------------------------------------------------------
    # 2. Check duplicate slot assignments
    # ---------------------------------------------------------

    duplicate_slots = (
        baseline_schedule_df
        .groupby("slot_id")
        .size()
    )

    for slot_id, count in duplicate_slots.items():

        if count > 1:

            violations.append({
                "slot_id": slot_id,
                "violation": "Duplicate slot assignment"
            })

    # ---------------------------------------------------------
    # 3. Check patient overlapping appointments
    # ---------------------------------------------------------

    duplicate_patient_slots = (
        baseline_schedule_df
        .groupby(
            ["patient_id", "date", "start_time"]
        )
        .size()
    )

    for key, count in duplicate_patient_slots.items():

        if count > 1:

            violations.append({
                "patient_id": key[0],
                "date": key[1],
                "start_time": key[2],
                "violation": "Patient time conflict"
            })

    # ---------------------------------------------------------
    # 4. Check doctor daily capacity
    # ---------------------------------------------------------

    doctor_daily_counts = (
        baseline_schedule_df
        .groupby(["doctor_id", "date"])
        .size()
        .reset_index(name="appointment_count")
    )

    for _, row in doctor_daily_counts.iterrows():

        doctor = doctors_df[
            doctors_df["doctor_id"] == row["doctor_id"]
        ].iloc[0]

        if row["appointment_count"] > doctor["max_daily_appointments"]:

            violations.append({
                "doctor_id": row["doctor_id"],
                "date": row["date"],
                "violation": "Daily capacity exceeded"
            })

    # ---------------------------------------------------------
    # 5. Check that all assigned slots exist
    # ---------------------------------------------------------

    valid_slot_ids = set(slots_df["slot_id"])

    for _, appointment in baseline_schedule_df.iterrows():

        if appointment["slot_id"] not in valid_slot_ids:

            violations.append({
                "patient_id": appointment["patient_id"],
                "slot_id": appointment["slot_id"],
                "violation": "Invalid slot"
            })

    return pd.DataFrame(violations)


# ---------------------------------------------------------
# Run validation
# ---------------------------------------------------------

baseline_violations_df = validate_baseline_schedule(
    baseline_schedule_df,
    doctors_df,
    slots_df
)


print("===== BASELINE VALIDATION =====")
print("Total scheduled appointments:", len(baseline_schedule_df))
print("Total constraint violations:", len(baseline_violations_df))
print()

if len(baseline_violations_df) == 0:
    print("✓ Baseline schedule is valid.")
else:
    print(baseline_violations_df)

===== BASELINE VALIDATION =====
Total scheduled appointments: 100
Total constraint violations: 0

✓ Baseline schedule is valid.


Validation Result

The final schedule contains 100 scheduled appointments and 0 constraint violations.

This confirms that all generated assignments satisfy the constraints implemented in the validation process.

Therefore, the 100% assignment rate achieved by the baseline is not the result of accepting invalid assignments; all scheduled appointments are valid according to the defined rules.

Baseline Performance Evaluation

This section calculates the main performance metrics of the Single-Agent baseline.

The evaluation measures both scheduling effectiveness and patient preference satisfaction. The metrics include successful assignment rate, constraint violations, preferred physician satisfaction, preferred date satisfaction, preferred time satisfaction, and date deviation.

These metrics provide quantitative evidence that can later be used to compare the Single-Agent baseline with the Multi-Agent System.

In [9]:
# Part 9 — Baseline Evaluation

def calculate_baseline_metrics(
    baseline_schedule_df,
    appointment_requests_df,
    baseline_violations_df
):

    total_patients = len(appointment_requests_df)

    scheduled_patients = len(baseline_schedule_df)

    unscheduled_patients = (
        total_patients - scheduled_patients
    )

    # ---------------------------------------------------------
    # Successful Assignment Rate
    # ---------------------------------------------------------

    successful_assignment_rate = (
        scheduled_patients / total_patients
    ) * 100

    # ---------------------------------------------------------
    # Preferred Doctor Satisfaction
    # ---------------------------------------------------------

    requests_with_preferred_doctor = (
        appointment_requests_df[
            appointment_requests_df["preferred_doctor_id"].notna()
        ]
    )

    if len(requests_with_preferred_doctor) > 0:

        doctor_check = baseline_schedule_df.merge(
            requests_with_preferred_doctor[
                ["request_id", "preferred_doctor_id"]
            ],
            on="request_id",
            how="inner"
        )

        preferred_doctor_satisfaction = (
            (
                doctor_check["doctor_id"]
                == doctor_check["preferred_doctor_id"]
            ).mean()
            * 100
        )

    else:
        preferred_doctor_satisfaction = 0

    # ---------------------------------------------------------
    # Preferred Date Satisfaction
    # ---------------------------------------------------------

    date_check = baseline_schedule_df.merge(
        appointment_requests_df[
            ["request_id", "preferred_date"]
        ],
        on="request_id",
        how="left"
    )

    preferred_date_satisfaction = (
        (
            date_check["date"]
            == date_check["preferred_date"]
        ).mean()
        * 100
    )

    # ---------------------------------------------------------
    # Preferred Time Satisfaction
    # ---------------------------------------------------------

    time_check = baseline_schedule_df.merge(
        appointment_requests_df[
            ["request_id", "preferred_time"]
        ],
        on="request_id",
        how="left"
    )

    def check_time(row):

        if row["preferred_time"] == "No Preference":
            return None

        return matches_time_preference(
            row["start_time"],
            row["preferred_time"]
        )

    time_results = time_check.apply(
        check_time,
        axis=1
    )

    preferred_time_satisfaction = (
        time_results.dropna().mean()
        * 100
    )

    # ---------------------------------------------------------
    # Date Deviation
    # ---------------------------------------------------------

    date_check["date_deviation"] = (
        pd.to_datetime(date_check["date"])
        - pd.to_datetime(date_check["preferred_date"])
    ).dt.days.abs()

    average_date_deviation = (
        date_check["date_deviation"].mean()
    )

    maximum_date_deviation = (
        date_check["date_deviation"].max()
    )

    # ---------------------------------------------------------
    # Constraint Violations
    # ---------------------------------------------------------

    constraint_violations = len(
        baseline_violations_df
    )

    # ---------------------------------------------------------
    # Create metrics dictionary
    # ---------------------------------------------------------

    metrics = {
        "Total Patients": total_patients,
        "Scheduled Patients": scheduled_patients,
        "Unscheduled Patients": unscheduled_patients,
        "Successful Assignment Rate (%)":
            successful_assignment_rate,
        "Constraint Violations":
            constraint_violations,
        "Preferred Doctor Satisfaction (%)":
            preferred_doctor_satisfaction,
        "Preferred Date Satisfaction (%)":
            preferred_date_satisfaction,
        "Preferred Time Satisfaction (%)":
        preferred_time_satisfaction,
        "Average Date Deviation (days)":
            average_date_deviation,
        "Maximum Date Deviation (days)":
            maximum_date_deviation
    }

    return metrics


# ---------------------------------------------------------
# Calculate Baseline Metrics
# ---------------------------------------------------------

baseline_metrics = calculate_baseline_metrics(
    baseline_schedule_df,
    appointment_requests_df,
    baseline_violations_df
)


print("===== BASELINE EVALUATION =====")

for metric, value in baseline_metrics.items():

    if isinstance(value, float):
        print(f"{metric}: {value:.2f}")
    else:
        print(f"{metric}: {value}")

===== BASELINE EVALUATION =====
Total Patients: 100
Scheduled Patients: 100
Unscheduled Patients: 0
Successful Assignment Rate (%): 100.00
Constraint Violations: 0
Preferred Doctor Satisfaction (%): 95.24
Preferred Date Satisfaction (%): 76.00
Preferred Time Satisfaction (%): 100.00
Average Date Deviation (days): 0.27
Maximum Date Deviation (days): 2


Baseline Evaluation Result

The Easy Single-Agent baseline achieved the following results:

• Successful Assignment Rate: 100.00%
• Constraint Violations: 0
• Preferred Doctor Satisfaction: 95.24%
• Preferred Date Satisfaction: 76.00%
• Preferred Time Satisfaction: 100.00%
• Average Date Deviation: 0.27 days
• Maximum Date Deviation: 2 days

The results show that the baseline was able to schedule every request without violating the defined constraints while achieving high satisfaction of patient preferences.

This experiment therefore establishes a successful initial baseline before increasing the dataset size and introducing dynamic scheduling events.

Save Baseline Results

This section saves the generated baseline outputs as separate CSV files.

Saving the results allows the final schedule, unscheduled requests, constraint violations, and evaluation metrics to be reused later without rerunning the entire experiment.

These saved outputs can also support the later comparison between the Single-Agent and Multi-Agent systems.

In [10]:
# Part 10 — Save Baseline Results

# ---------------------------------------------------------
# 1. Save the final schedule
# ---------------------------------------------------------

baseline_schedule_df.to_csv(
    "baseline_schedule.csv",
    index=False
)

# ---------------------------------------------------------
# 2. Save unscheduled requests
# ---------------------------------------------------------

unscheduled_df.to_csv(
    "baseline_unscheduled.csv",
    index=False
)

# ---------------------------------------------------------
# 3. Save validation violations
# ---------------------------------------------------------

baseline_violations_df.to_csv(
    "baseline_violations.csv",
    index=False
)

# ---------------------------------------------------------
# 4. Save evaluation metrics
# ---------------------------------------------------------

baseline_metrics_df = pd.DataFrame(
    list(baseline_metrics.items()),
    columns=["Metric", "Value"]
)

baseline_metrics_df.to_csv(
    "baseline_metrics.csv",
    index=False
)


print("===== BASELINE RESULTS SAVED =====")
print("✓ baseline_schedule.csv")
print("✓ baseline_unscheduled.csv")
print("✓ baseline_violations.csv")
print("✓ baseline_metrics.csv")

===== BASELINE RESULTS SAVED =====
✓ baseline_schedule.csv
✓ baseline_unscheduled.csv
✓ baseline_violations.csv
✓ baseline_metrics.csv


Saved Results
The baseline results were successfully saved as:
baseline_schedule.csv
baseline_unscheduled.csv
baseline_violations.csv
baseline_metrics.csv
These files preserve the outputs of the Easy Single-Agent experiment for later analysis and comparison.

## M4 Live Demo Verification — Easy / Static Baseline

This verification block is part of the full project notebook and is also copied **unchanged** into the simplified M4 demo notebook.
It verifies the same Easy / Static baseline used in the live 5-minute demonstration.


In [11]:
# M4 — End-to-End Baseline Verification
m4_total_requests = len(appointment_requests_df)
m4_scheduled = len(baseline_schedule_df)
m4_unscheduled = len(unscheduled_df)
m4_violations = len(baseline_violations_df)
m4_assignment_rate = (m4_scheduled / m4_total_requests * 100) if m4_total_requests else 0

m4_end_to_end_success = (
    m4_total_requests == 100
    and m4_scheduled == 100
    and m4_unscheduled == 0
    and m4_violations == 0
)

print("===== M4 END-TO-END VERIFICATION =====")
print(f"End-to-end task completed: {m4_end_to_end_success}")
print(f"Total requests: {m4_total_requests}")
print(f"Scheduled: {m4_scheduled}")
print(f"Unscheduled: {m4_unscheduled}")
print(f"Constraint violations: {m4_violations}")
print(f"Assignment rate: {m4_assignment_rate:.2f}%")
assert m4_end_to_end_success, "M4 baseline verification failed."
print("Baseline valid: True")


===== M4 END-TO-END VERIFICATION =====
End-to-end task completed: True
Total requests: 100
Scheduled: 100
Unscheduled: 0
Constraint violations: 0
Assignment rate: 100.00%
Baseline valid: True


Experiment 2 — Increased Complexity and Dynamic Scheduling

The second experiment extends the initial Easy baseline into a larger and more challenging hospital scheduling environment.

The purpose of this experiment is to examine the behaviour of the Single-Agent scheduler when the number of resources and appointment requests increases and when the scheduling environment becomes dynamic.

The experiment introduces several sources of complexity:

• A larger physician dataset
• A larger patient population
• Multiple appointment requests for some patients
• 30-minute and 60-minute appointment durations
• Physician appointment and time capacity constraints
• Maximum waiting-day constraints
• Patient cancellations
• Physician unavailability periods
• New appointment requests
• Physician schedule changes

The initial schedule is first generated and evaluated under static conditions. Afterward, dynamic events are applied sequentially to simulate changes that may occur in a real hospital environment.

The objective is to evaluate whether the Single-Agent baseline can maintain a valid and effective schedule while adapting to these changes.

The same experimental environment will later be used by the Multi-Agent System to ensure a controlled comparison.

M1 — Expanded Doctors Dataset

This section increases the size and diversity of the physician dataset.

The experiment uses 40 physicians distributed across 10 medical specialties. In addition to specialty and experience, physicians have different working hours and capacity limits measured by both the number of appointments and total appointment minutes per day.

These additional resource constraints make the scheduling problem more realistic because a physician may have available time slots but still be unable to accept additional appointments after reaching the defined daily capacity.

In [12]:
import random
import pandas as pd
from datetime import datetime, timedelta

# ============================================================
# M1 — EXPANDED DOCTORS DATASET
# ============================================================

MEDIUM_RANDOM_SEED = 42
random.seed(MEDIUM_RANDOM_SEED)

# Existing: 20 doctors
# Experiment 2: 40 doctors
MEDIUM_DOCTOR_COUNT = 40

MEDIUM_SPECIALTIES = [
    "Cardiology",
    "Dermatology",
    "Neurology",
    "Pediatrics",
    "Orthopedics",
    "General Medicine",
    "Internal Medicine",
    "ENT",
    "Ophthalmology",
    "Obstetrics and Gynecology"
]

# Four doctors per specialty
DOCTORS_PER_SPECIALTY = 4

WORK_SHIFTS = [
    ("08:00", "14:00"),
    ("09:00", "15:00"),
    ("10:00", "16:00")
]


def generate_medium_doctors():
    doctors = []
    doctor_number = 1

    for specialty in MEDIUM_SPECIALTIES:

        for _ in range(DOCTORS_PER_SPECIALTY):

            work_start, work_end = random.choice(WORK_SHIFTS)

            experience_years = random.randint(3, 20)

            # Slightly tighter capacity than the old Medium
            max_daily_appointments = random.choice([3, 4, 5])

            # New resource constraint.
            # A 30-minute appointment consumes 30 minutes;
            # a 60-minute appointment consumes 60 minutes.
            max_daily_minutes = random.choice([180, 210, 240])

            doctors.append({
                "doctor_id": f"D{doctor_number:03d}",
                "specialty": specialty,
                "experience_years": experience_years,
                "work_start": work_start,
                "work_end": work_end,
                "max_daily_appointments": max_daily_appointments,
                "max_daily_minutes": max_daily_minutes
            })

            doctor_number += 1

    return pd.DataFrame(doctors)


medium_doctors_df = generate_medium_doctors()

print("===== EXPERIMENT 2 — M1 DOCTORS =====")
print("Number of doctors:", len(medium_doctors_df))
print("Number of specialties:", medium_doctors_df["specialty"].nunique())
print("Doctors per specialty:", DOCTORS_PER_SPECIALTY)
print()
print(medium_doctors_df.head(10))

===== EXPERIMENT 2 — M1 DOCTORS =====
Number of doctors: 40
Number of specialties: 10
Doctors per specialty: 4

  doctor_id    specialty  ...  max_daily_appointments max_daily_minutes
0      D001   Cardiology  ...                       3               240
1      D002   Cardiology  ...                       3               180
2      D003   Cardiology  ...                       5               240
3      D004   Cardiology  ...                       5               210
4      D005  Dermatology  ...                       3               180
5      D006  Dermatology  ...                       5               180
6      D007  Dermatology  ...                       5               240
7      D008  Dermatology  ...                       4               180
8      D009    Neurology  ...                       3               180
9      D010    Neurology  ...                       4               210

[10 rows x 7 columns]


M1 Result

The expanded dataset contains:

• 40 physicians
• 10 medical specialties
• 4 physicians per specialty

Physicians have different working hours and daily capacity limits.

The larger and more heterogeneous physician dataset increases the number of possible assignments while also introducing stronger resource constraints for the scheduler.

M2 — Expanded Patient Dataset

This section generates a substantially larger patient population for Experiment 2.

Each patient has a required specialty, optional preferred physician, preferred date and time, and a maximum acceptable waiting period. Some patients also require a second appointment with another specialty.

Introducing multiple appointments per patient increases the complexity of the problem because the scheduler must consider the patient’s appointments collectively and avoid conflicts between them.

In [13]:
# ============================================================
# M2 — EXPANDED PATIENT DATASET
# ============================================================

# Existing: 100 patients / 14 days
# Experiment 2: 2200 patients / 21 days
MEDIUM_PATIENT_COUNT = 2200
MEDIUM_START_DATE = datetime(2026, 9, 14)
MEDIUM_SCHEDULING_DAYS = 21

PREFERRED_TIME_OPTIONS = [
    "Morning",
    "Afternoon",
    "No Preference"
]

# 35% of patients require a second appointment.
MULTI_APPOINTMENT_PROBABILITY = 0.50

# Maximum allowed waiting time for an appointment request.
MAX_WAITING_DAYS_OPTIONS = [3, 5, 7]

# 30% of requests require a 60-minute appointment.
LONG_APPOINTMENT_PROBABILITY = 0.40


def generate_medium_patients(
    num_patients,
    doctors_df,
    start_date
):
    patients = []

    for i in range(1, num_patients + 1):

        patient_id = f"P{i:04d}"

        # Primary specialty
        required_specialty = random.choice(
            MEDIUM_SPECIALTIES
        )

        # Around 40% have a preferred doctor
        specialty_doctors = doctors_df[
            doctors_df["specialty"] == required_specialty
        ]["doctor_id"].tolist()

        if random.random() < 0.40:
            preferred_doctor_id = random.choice(
                specialty_doctors
            )
        else:
            preferred_doctor_id = None

        preferred_date = start_date + timedelta(
            days=random.randint(
                0,
                MEDIUM_SCHEDULING_DAYS - 1
            )
        )

        preferred_time = random.choice(
            PREFERRED_TIME_OPTIONS
        )

        max_waiting_days = random.choice(
            MAX_WAITING_DAYS_OPTIONS
        )

        # Some patients require a second appointment
        # with another specialty.
        requires_second_appointment = (
            random.random()
            < MULTI_APPOINTMENT_PROBABILITY
        )

        second_specialty = None

        if requires_second_appointment:
            second_specialty_options = [
                s for s in MEDIUM_SPECIALTIES
                if s != required_specialty
            ]

            second_specialty = random.choice(
                second_specialty_options
            )

        patients.append({
            "patient_id": patient_id,
            "required_specialty": required_specialty,
            "preferred_doctor_id": preferred_doctor_id,
            "preferred_date": preferred_date.strftime("%Y-%m-%d"),
            "preferred_time": preferred_time,
            "max_waiting_days": max_waiting_days,
            "requires_second_appointment": requires_second_appointment,
            "second_specialty": second_specialty
        })

    return pd.DataFrame(patients)


medium_patients_df = generate_medium_patients(
    MEDIUM_PATIENT_COUNT,
    medium_doctors_df,
    MEDIUM_START_DATE
)

print()
print("===== EXPERIMENT 2 — M2 PATIENTS =====")
print("Number of patients:", len(medium_patients_df))
print("Scheduling days:", MEDIUM_SCHEDULING_DAYS)
print(
    "Patients requiring a second appointment:",
    medium_patients_df["requires_second_appointment"].sum()
)
print()
print(medium_patients_df.head(10))



===== EXPERIMENT 2 — M2 PATIENTS =====
Number of patients: 2200
Scheduling days: 21
Patients requiring a second appointment: 1098

  patient_id required_specialty  ... requires_second_appointment second_specialty
0      P0001          Neurology  ...                       False             None
1      P0002        Dermatology  ...                       False             None
2      P0003          Neurology  ...                       False             None
3      P0004                ENT  ...                        True      Dermatology
4      P0005        Dermatology  ...                        True    Ophthalmology
5      P0006      Ophthalmology  ...                       False             None
6      P0007        Orthopedics  ...                        True       Pediatrics
7      P0008        Dermatology  ...                        True       Cardiology
8      P0009         Pediatrics  ...                        True    Ophthalmology
9      P0010         Pediatrics  ...            

M2 Result

The experiment contains 2,200 patients over a 21-day scheduling period.

Among these patients, 1,098 require a second appointment.

This means the scheduler must handle patients with either one or multiple appointment requirements, while maintaining compatibility between their appointments and respecting the available hospital resources.

M3 — Multiple Appointment Requests

This section converts the expanded patient requirements into individual appointment requests.

Patients requiring a second appointment generate an additional request with its own medical specialty and scheduling requirements.

Separating the requests from the patient records allows each appointment to be scheduled independently while still preserving the patient identifier and appointment number. This structure is important for detecting scheduling conflicts between multiple appointments belonging to the same patient.

In [14]:
# ============================================================
# M3 — MULTIPLE APPOINTMENT REQUESTS
# ============================================================

def generate_medium_appointment_requests(
    patients_df
):
    requests = []

    for _, patient in patients_df.iterrows():

        # Every patient gets a primary request.
        request_number = len(requests) + 1

        primary_duration = (
            60
            if random.random() < LONG_APPOINTMENT_PROBABILITY
            else 30
        )

        requests.append({
            "request_id": f"R{request_number:04d}",
            "patient_id": patient["patient_id"],
            "appointment_number": 1,
            "required_specialty": patient["required_specialty"],
            "preferred_doctor_id": patient["preferred_doctor_id"],
            "preferred_date": patient["preferred_date"],
            "preferred_time": patient["preferred_time"],
            "max_waiting_days": patient["max_waiting_days"],
            "appointment_duration": primary_duration
        })

        # Some patients receive a second request.
        if patient["requires_second_appointment"]:

            request_number = len(requests) + 1

            second_duration = (
                60
                if random.random() < LONG_APPOINTMENT_PROBABILITY
                else 30
            )

            requests.append({
                "request_id": f"R{request_number:04d}",
                "patient_id": patient["patient_id"],
                "appointment_number": 2,
                "required_specialty": patient["second_specialty"],
                # No artificial preferred doctor is assigned
                # for the second specialty.
                "preferred_doctor_id": None,
                "preferred_date": patient["preferred_date"],
                "preferred_time": patient["preferred_time"],
                "max_waiting_days": patient["max_waiting_days"],
                "appointment_duration": second_duration
            })

    return pd.DataFrame(requests)


medium_appointment_requests_df = (
    generate_medium_appointment_requests(
        medium_patients_df
    )
)

print()
print("===== EXPERIMENT 2 — M3 REQUESTS =====")
print(
    "Number of appointment requests:",
    len(medium_appointment_requests_df)
)
print(
    "Patients with multiple requests:",
    medium_appointment_requests_df[
        "patient_id"
    ].value_counts().gt(1).sum()
)
print()
print(
    medium_appointment_requests_df.head(10)
)



===== EXPERIMENT 2 — M3 REQUESTS =====
Number of appointment requests: 3298
Patients with multiple requests: 1098

  request_id patient_id  ...  max_waiting_days appointment_duration
0      R0001      P0001  ...                 3                   30
1      R0002      P0002  ...                 7                   30
2      R0003      P0003  ...                 3                   30
3      R0004      P0004  ...                 3                   30
4      R0005      P0004  ...                 3                   60
5      R0006      P0005  ...                 3                   60
6      R0007      P0005  ...                 3                   30
7      R0008      P0006  ...                 7                   60
8      R0009      P0007  ...                 5                   60
9      R0010      P0007  ...                 5                   30

[10 rows x 9 columns]


M3 Result

The expanded patient dataset produced 3,298 appointment requests.

A total of 1,098 patients have multiple appointment requests.

The increase from 2,200 patients to 3,298 requests demonstrates that the scheduling problem is no longer a simple one-patient/one-appointment problem. The scheduler must now manage multiple requests while avoiding conflicts and respecting the available resources.

M4 — Time Slots and Doctor Availability

This section generates the time-slot environment for the expanded experiment.

The base scheduling unit is reduced to 30 minutes, allowing the system to represent both 30-minute and 60-minute appointments.

A 30-minute appointment requires one base slot, while a 60-minute appointment requires two consecutive base slots. This representation provides greater flexibility while allowing the scheduler to verify that longer appointments have sufficient consecutive availability.

In [15]:
# ============================================================
# M4 — TIME SLOTS & DOCTOR AVAILABILITY
# ============================================================

MEDIUM_SLOT_DURATION_MINUTES = 30


def generate_medium_time_slots(
    doctors_df,
    start_date,
    scheduling_days
):

    slots = []

    for _, doctor in doctors_df.iterrows():

        doctor_id = doctor["doctor_id"]

        start_hour, start_minute = map(
            int,
            doctor["work_start"].split(":")
        )

        end_hour, end_minute = map(
            int,
            doctor["work_end"].split(":")
        )

        for day_offset in range(scheduling_days):

            current_date = (
                start_date
                + timedelta(days=day_offset)
            )

            current_time = datetime.combine(
                current_date,
                datetime.min.time()
            ).replace(
                hour=start_hour,
                minute=start_minute
            )

            end_time = datetime.combine(
                current_date,
                datetime.min.time()
            ).replace(
                hour=end_hour,
                minute=end_minute
            )

            while current_time < end_time:

                slot_end_time = (
                    current_time
                    + timedelta(
                        minutes=MEDIUM_SLOT_DURATION_MINUTES
                    )
                )

                slots.append({
                    "slot_id": f"S{len(slots) + 1:06d}",
                    "doctor_id": doctor_id,
                    "date": current_date.strftime("%Y-%m-%d"),
                    "start_time": current_time.strftime("%H:%M"),
                    "end_time": slot_end_time.strftime("%H:%M"),
                    "status": "Available"
                })

                current_time = slot_end_time

    return pd.DataFrame(slots)


medium_slots_df = generate_medium_time_slots(
    medium_doctors_df,
    MEDIUM_START_DATE,
    MEDIUM_SCHEDULING_DAYS
)

print()
print("===== EXPERIMENT 2 — M4 TIME SLOTS =====")
print("Number of doctors:", len(medium_doctors_df))
print("Scheduling days:", MEDIUM_SCHEDULING_DAYS)
print("Base slot duration:", MEDIUM_SLOT_DURATION_MINUTES)
print("Total slots:", len(medium_slots_df))
print()
print(medium_slots_df.head(10))



===== EXPERIMENT 2 — M4 TIME SLOTS =====
Number of doctors: 40
Scheduling days: 21
Base slot duration: 30
Total slots: 10080

   slot_id doctor_id        date start_time end_time     status
0  S000001      D001  2026-09-14      10:00    10:30  Available
1  S000002      D001  2026-09-14      10:30    11:00  Available
2  S000003      D001  2026-09-14      11:00    11:30  Available
3  S000004      D001  2026-09-14      11:30    12:00  Available
4  S000005      D001  2026-09-14      12:00    12:30  Available
5  S000006      D001  2026-09-14      12:30    13:00  Available
6  S000007      D001  2026-09-14      13:00    13:30  Available
7  S000008      D001  2026-09-14      13:30    14:00  Available
8  S000009      D001  2026-09-14      14:00    14:30  Available
9  S000010      D001  2026-09-14      14:30    15:00  Available


M4 Result

The expanded scheduling environment contains:

• 40 physicians
• 21 scheduling days
• 30-minute base slots
• 10,080 generated slots

Using 30-minute base slots allows the system to support appointments with different durations while maintaining precise control over physician availability.

M5 — Initial Single-Agent Scheduling

This section runs the optimized Single-Agent scheduler on the expanded dataset before any dynamic events occur.

The scheduler evaluates all 3,298 appointment requests and searches for feasible assignments while considering specialty compatibility, slot availability, physician capacity, appointment duration, patient conflicts, preferences, and maximum waiting periods.

The resulting schedule represents the static baseline state that will later be exposed to dynamic events.

In [16]:
# ============================================================
# ============================================================
# M5 — INITIAL SINGLE-AGENT SCHEDULING (OPTIMIZED)
# ============================================================

# The scheduling logic is unchanged. The implementation below
# avoids repeatedly parsing dates/times, scanning the whole schedule,
# and rebuilding consecutive-slot combinations inside the inner loop.

MEDIUM_SLOT_DURATION_MINUTES = 30


def time_to_minutes(time_string):
    hour, minute = map(int, time_string.split(":"))
    return hour * 60 + minute


def matches_medium_time_preference(start_time, preferred_time):
    hour = int(start_time[:2])
    if preferred_time == "Morning":
        return hour < 12
    if preferred_time == "Afternoon":
        return hour >= 12
    return True


def calculate_medium_assignment_score_fast(
    preferred_doctor_id,
    preferred_date,
    preferred_time,
    doctor_id,
    slot_date,
    slot_start_hour,
    date_difference
):
    score = 0

    if preferred_doctor_id is not None and preferred_doctor_id == doctor_id:
        score += 40

    if slot_date == preferred_date:
        score += 30

    if preferred_time != "No Preference":
        if preferred_time == "Morning" and slot_start_hour < 12:
            score += 20
        elif preferred_time == "Afternoon" and slot_start_hour >= 12:
            score += 20

    score += max(0, 10 - date_difference)
    return score


def get_required_slot_keys(
    doctor_id,
    date,
    start_time,
    duration_minutes,
    slots_by_key
):
    """Return the consecutive 30-minute base-slot keys."""
    duration_slots = int(duration_minutes) // MEDIUM_SLOT_DURATION_MINUTES
    start_min = time_to_minutes(start_time)
    required_keys = []

    for i in range(duration_slots):
        slot_start_min = start_min + i * MEDIUM_SLOT_DURATION_MINUTES
        slot_end_min = slot_start_min + MEDIUM_SLOT_DURATION_MINUTES
        slot_start = f"{slot_start_min // 60:02d}:{slot_start_min % 60:02d}"
        slot_end = f"{slot_end_min // 60:02d}:{slot_end_min % 60:02d}"
        key = (doctor_id, date, slot_start, slot_end)
        if key not in slots_by_key:
            return None
        required_keys.append(key)

    return required_keys


def run_medium_baseline(
    appointment_requests_df,
    doctors_df,
    slots_df
):
    """
    Optimized implementation of the same Medium single-agent
    scheduling policy.

    Important: no patient priority is used.
    """

    schedule = []
    unscheduled = []

    # --------------------------------------------------------
    # 1. Fast doctor lookup
    # --------------------------------------------------------
    doctors_lookup = {}
    for row in doctors_df.to_dict("records"):
        doctors_lookup[row["doctor_id"]] = row

    # --------------------------------------------------------
    # 2. Precompute slot information once.
    #    Each candidate already contains the required 30/60-min
    #    block, so M5 does not reconstruct it for every request.
    # --------------------------------------------------------
    slots_by_specialty_duration = {}
    slot_lookup = {}

    for specialty in MEDIUM_SPECIALTIES:
        slots_by_specialty_duration[(specialty, 30)] = []
        slots_by_specialty_duration[(specialty, 60)] = []

    for row in slots_df.to_dict("records"):
        doctor_id = row["doctor_id"]
        doctor = doctors_lookup[doctor_id]
        date = row["date"]
        start_time = row["start_time"]
        end_time = row["end_time"]

        key = (doctor_id, date, start_time, end_time)
        slot_lookup[key] = row

        start_min = time_to_minutes(start_time)
        end_min = time_to_minutes(end_time)

        candidate_base = {
            "doctor_id": doctor_id,
            "doctor": doctor,
            "date": date,
            "start_time": start_time,
            "start_min": start_min,
            "start_hour": start_min // 60,
            "end_time": end_time,
            "end_min": end_min,
            "slot_key": key,
        }

        slots_by_specialty_duration[(doctor["specialty"], 30)].append(
            (candidate_base, (key,))
        )

    # Build valid 60-minute blocks from consecutive base slots.
    # Only 30-minute grid continuity is accepted.
    for specialty in MEDIUM_SPECIALTIES:
        base_candidates = slots_by_specialty_duration[(specialty, 30)]
        base_by_key = {
            item[0]["slot_key"]: item[0]
            for item in base_candidates
        }

        for base, _ in base_candidates:
            first_key = base["slot_key"]
            next_start_min = base["start_min"] + 30
            next_start = f"{next_start_min // 60:02d}:{next_start_min % 60:02d}"
            next_key = (
                base["doctor_id"],
                base["date"],
                next_start,
                f"{(next_start_min + 30) // 60:02d}:{(next_start_min + 30) % 60:02d}"
            )

            second = base_by_key.get(next_key)
            if second is None:
                continue

            block = dict(base)
            block["end_time"] = second["end_time"]
            block["end_min"] = second["end_min"]
            slots_by_specialty_duration[(specialty, 60)].append(
                (block, (first_key, next_key))
            )

    # --------------------------------------------------------
    # 3. Convert requests to records and precompute dates.
    # --------------------------------------------------------
    request_records = []
    for request in appointment_requests_df.to_dict("records"):
        preferred_date_obj = datetime.strptime(
            request["preferred_date"], "%Y-%m-%d"
        ).date()
        request["_preferred_date_obj"] = preferred_date_obj
        request["_preferred_doctor_id"] = request["preferred_doctor_id"]
        request_records.append(request)

    # --------------------------------------------------------
    # 4. Incremental state for O(1)-average capacity and overlap
    #    checks. This replaces repeated scans of the schedule.
    # --------------------------------------------------------
    booked_slots = set()
    doctor_daily_counts = {}
    doctor_daily_minutes = {}

    # patient_intervals[(patient_id, date)] = [(start_min, end_min), ...]
    patient_intervals = {}

    for request in request_records:
        duration = int(request["appointment_duration"])
        specialty = request["required_specialty"]
        candidates = slots_by_specialty_duration.get(
            (specialty, duration), []
        )

        preferred_date_obj = request["_preferred_date_obj"]
        max_waiting_days = int(request["max_waiting_days"])
        preferred_doctor_id = request["preferred_doctor_id"]
        preferred_time = request["preferred_time"]
        patient_id = request["patient_id"]

        best = None
        best_sort_key = None

        # ----------------------------------------------------
        # 5. Scan only valid specialty + duration candidates.
        #    No DataFrame filtering and no datetime parsing occur
        #    inside this loop.
        # ----------------------------------------------------
        for candidate, required_keys in candidates:
            doctor_id = candidate["doctor_id"]
            date = candidate["date"]
            start_min = candidate["start_min"]
            end_min = candidate["end_min"]

            if any(key in booked_slots for key in required_keys):
                continue

            capacity_key = (doctor_id, date)
            current_count = doctor_daily_counts.get(capacity_key, 0)
            doctor = candidate["doctor"]

            if current_count >= int(doctor["max_daily_appointments"]):
                continue

            current_minutes = doctor_daily_minutes.get(capacity_key, 0)
            if current_minutes + duration > int(doctor["max_daily_minutes"]):
                continue

            scheduled_date = datetime.strptime(date, "%Y-%m-%d").date()
            date_difference = abs((scheduled_date - preferred_date_obj).days)

            if date_difference > max_waiting_days:
                continue

            # Patient overlap check uses only appointments already
            # assigned to this patient on this date.
            intervals = patient_intervals.get((patient_id, date), [])
            overlap = False
            for existing_start, existing_end in intervals:
                if start_min < existing_end and end_min > existing_start:
                    overlap = True
                    break

            if overlap:
                continue

            score = calculate_medium_assignment_score_fast(
                preferred_doctor_id,
                request["preferred_date"],
                preferred_time,
                doctor_id,
                date,
                candidate["start_hour"],
                date_difference
            )

            # Same tie-breaking policy as before:
            # highest score -> earliest date -> earliest start time.
            sort_key = (-score, date, candidate["start_time"])

            if best is None or sort_key < best_sort_key:
                best = (candidate, required_keys, score, capacity_key)
                best_sort_key = sort_key

        if best is not None:
            candidate, required_keys, score, capacity_key = best
            doctor = candidate["doctor"]
            date = candidate["date"]
            start_time = candidate["start_time"]
            end_time = candidate["end_time"]

            schedule.append({
                "request_id": request["request_id"],
                "patient_id": patient_id,
                "appointment_number": request["appointment_number"],
                "doctor_id": doctor["doctor_id"],
                "required_specialty": request["required_specialty"],
                "date": date,
                "start_time": start_time,
                "end_time": end_time,
                "appointment_duration": duration,
                "preferred_doctor_id": request["preferred_doctor_id"],
                "preferred_date": request["preferred_date"],
                "preferred_time": request["preferred_time"],
                "max_waiting_days": request["max_waiting_days"],
                "score": score
            })

            booked_slots.update(required_keys)
            doctor_daily_counts[capacity_key] = (
                doctor_daily_counts.get(capacity_key, 0) + 1
            )
            doctor_daily_minutes[capacity_key] = (
                doctor_daily_minutes.get(capacity_key, 0) + duration
            )
            patient_intervals.setdefault((patient_id, date), []).append(
                (candidate["start_min"], candidate["end_min"])
            )

        else:
            unscheduled.append({
                "request_id": request["request_id"],
                "patient_id": patient_id,
                "appointment_number": request["appointment_number"],
                "required_specialty": specialty,
                "appointment_duration": duration
            })

    return pd.DataFrame(schedule), pd.DataFrame(unscheduled)


medium_baseline_schedule_df, medium_baseline_unscheduled_df = (
    run_medium_baseline(
        medium_appointment_requests_df,
        medium_doctors_df,
        medium_slots_df
    )
)

print()
print("===== EXPERIMENT 2 — M5 INITIAL BASELINE =====")
print("Total appointment requests:", len(medium_appointment_requests_df))
print("Scheduled appointments:", len(medium_baseline_schedule_df))
print("Unscheduled appointments:", len(medium_baseline_unscheduled_df))



===== EXPERIMENT 2 — M5 INITIAL BASELINE =====
Total appointment requests: 3298
Scheduled appointments: 3113
Unscheduled appointments: 185


M5 Result

The initial Single-Agent scheduler processed 3,298 appointment requests.

It successfully scheduled 3,113 appointments, while 185 requests remained unscheduled.

The resulting assignment rate is approximately 94.39%.

Unlike the Easy experiment, the larger number of requests and additional constraints create resource competition, making it impossible to satisfy every request.

M6 — Initial Baseline Validation

This section validates the initial schedule produced by the Single-Agent system.

The validation checks whether the scheduled appointments satisfy the expanded set of constraints, including specialty compatibility, valid time slots, physician capacity, appointment duration, and patient scheduling conflicts.

The purpose is to verify that the unscheduled requests are caused by resource limitations rather than by accepting invalid assignments.

In [17]:
# M6 — INITIAL BASELINE VALIDATION
# ============================================================

def validate_medium_baseline_schedule(schedule_df, doctors_df, slots_df):
    """Validate the Medium baseline without repeated DataFrame scans."""
    violations = []
    doctors_lookup = {r["doctor_id"]: r for r in doctors_df.to_dict("records")}
    valid_slots = {(r["doctor_id"], r["date"], r["start_time"], r["end_time"]) for r in slots_df.to_dict("records")}
    slots_by_key = {k: True for k in valid_slots}
    doctor_daily_counts = {}
    doctor_daily_minutes = {}
    doctor_slot_assignments = set()
    patient_groups = {}

    for appointment in schedule_df.to_dict("records"):
        doctor_id = appointment["doctor_id"]; patient_id = appointment["patient_id"]
        date = appointment["date"]; start_time = appointment["start_time"]
        duration = int(appointment["appointment_duration"])

        if doctor_id not in doctors_lookup:
            violations.append({"patient_id": patient_id, "doctor_id": doctor_id, "violation": "Doctor does not exist"})
            continue
        doctor = doctors_lookup[doctor_id]

        if doctor["specialty"] != appointment["required_specialty"]:
            violations.append({"patient_id": patient_id, "doctor_id": doctor_id, "violation": "Specialty mismatch"})

        required_keys = get_required_slot_keys(doctor_id, date, start_time, duration, slots_by_key)
        if required_keys is None:
            violations.append({"patient_id": patient_id, "doctor_id": doctor_id, "violation": "Invalid or non-consecutive appointment slots"})
        else:
            for slot_key in required_keys:
                if slot_key in doctor_slot_assignments:
                    violations.append({"patient_id": patient_id, "doctor_id": doctor_id, "violation": "Doctor double booking"})
                doctor_slot_assignments.add(slot_key)

        capacity_key = (doctor_id, date)
        count = doctor_daily_counts.get(capacity_key, 0) + 1
        doctor_daily_counts[capacity_key] = count
        if count > int(doctor["max_daily_appointments"]):
            violations.append({"patient_id": patient_id, "doctor_id": doctor_id, "violation": "Doctor daily appointment capacity exceeded"})

        minutes = doctor_daily_minutes.get(capacity_key, 0) + duration
        doctor_daily_minutes[capacity_key] = minutes
        if minutes > int(doctor["max_daily_minutes"]):
            violations.append({"patient_id": patient_id, "doctor_id": doctor_id, "violation": "Doctor daily minutes exceeded"})

        patient_groups.setdefault((patient_id, date), []).append(appointment)

    for (patient_id, date), appointments in patient_groups.items():
        if len(appointments) < 2: continue
        ordered = sorted(appointments, key=lambda x: time_to_minutes(x["start_time"]))
        previous_end = time_to_minutes(ordered[0]["end_time"])
        previous_doctor = ordered[0]["doctor_id"]
        for current in ordered[1:]:
            current_start = time_to_minutes(current["start_time"])
            if previous_end > current_start:
                violations.append({"patient_id": patient_id, "doctor_id": previous_doctor, "violation": "Patient overlapping appointments"})
            current_end = time_to_minutes(current["end_time"])
            if current_end > previous_end:
                previous_end = current_end; previous_doctor = current["doctor_id"]

    return pd.DataFrame(violations)


M6 Result

The initial baseline contains 0 constraint violations.

Therefore, all 3,113 scheduled appointments satisfy the implemented scheduling constraints.

The 185 unscheduled requests were not caused by invalid scheduling decisions; instead, they remained unscheduled because no feasible assignment satisfying the constraints was available.

M7 — Initial Baseline Evaluation

This section evaluates the performance of the initial Single-Agent schedule using the expanded set of metrics.

In addition to assignment and preference satisfaction, the evaluation considers the total number of scheduled minutes and physician capacity utilization. These measures provide information about how effectively the available hospital resources are being used.

In [18]:
# M7 — INITIAL BASELINE EVALUATION
# ============================================================

def evaluate_medium_initial_baseline(
    requests_df,
    schedule_df,
    unscheduled_df,
    violations_df
):

    total_requests = len(requests_df)
    scheduled_requests = len(schedule_df)
    unscheduled_requests = len(unscheduled_df)

    assignment_rate = (
        scheduled_requests
        / total_requests
        * 100
        if total_requests > 0
        else 0
    )

    constraint_violations = len(
        violations_df
    )

    # Preference satisfaction
    if scheduled_requests > 0:

        preferred_doctor_requests = (
            schedule_df[
                schedule_df["preferred_doctor_id"].notna()
            ]
        )

        if len(preferred_doctor_requests) > 0:
            preferred_doctor_satisfaction = (
                (
                    preferred_doctor_requests["doctor_id"]
                    ==
                    preferred_doctor_requests[
                        "preferred_doctor_id"
                    ]
                ).mean()
                * 100
            )
        else:
            preferred_doctor_satisfaction = 0

        preferred_date_satisfaction = (
            (
                schedule_df["date"]
                ==
                schedule_df["preferred_date"]
            ).mean()
            * 100
        )

        time_preference_requests = (
            schedule_df[
                schedule_df["preferred_time"]
                != "No Preference"
            ]
        )

        if len(time_preference_requests) > 0:
            preferred_time_satisfaction = (
                time_preference_requests.apply(
                    lambda row:
                        matches_medium_time_preference(
                            row["start_time"],
                            row["preferred_time"]
                        ),
                    axis=1
                ).mean()
                * 100
            )
        else:
            preferred_time_satisfaction = 0

    else:
        preferred_doctor_satisfaction = 0
        preferred_date_satisfaction = 0
        preferred_time_satisfaction = 0

    # Date deviation
    date_deviations = []

    for _, row in schedule_df.iterrows():

        scheduled_date = datetime.strptime(
            row["date"],
            "%Y-%m-%d"
        ).date()

        preferred_date = datetime.strptime(
            row["preferred_date"],
            "%Y-%m-%d"
        ).date()

        date_deviations.append(
            abs(
                (scheduled_date - preferred_date).days
            )
        )

    average_date_deviation = (
        sum(date_deviations)
        / len(date_deviations)
        if date_deviations
        else 0
    )

    maximum_date_deviation = (
        max(date_deviations)
        if date_deviations
        else 0
    )

    # Doctor utilization by appointment minutes.
    total_scheduled_minutes = (
        schedule_df["appointment_duration"].sum()
        if not schedule_df.empty
        else 0
    )

    total_available_minutes = 0

    for _, doctor in medium_doctors_df.iterrows():
        total_available_minutes += (
            doctor["max_daily_minutes"]
            * MEDIUM_SCHEDULING_DAYS
        )

    doctor_utilization = (
        total_scheduled_minutes
        / total_available_minutes
        * 100
        if total_available_minutes > 0
        else 0
    )

    return {
        "Total Appointment Requests":
            total_requests,

        "Scheduled Appointments":
            scheduled_requests,

        "Unscheduled Appointments":
            unscheduled_requests,

        "Successful Assignment Rate (%)":
            round(assignment_rate, 2),

        "Constraint Violations":
            constraint_violations,

        "Preferred Doctor Satisfaction (%)":
            round(preferred_doctor_satisfaction, 2),

        "Preferred Date Satisfaction (%)":
            round(preferred_date_satisfaction, 2),

        "Preferred Time Satisfaction (%)":
            round(preferred_time_satisfaction, 2),

        "Average Date Deviation (days)":
            round(average_date_deviation, 2),

        "Maximum Date Deviation (days)":
            maximum_date_deviation,

        "Total Scheduled Minutes":
            int(total_scheduled_minutes),

        "Doctor Capacity Utilization (%)":
            round(doctor_utilization, 2)
    }


medium_baseline_violations_df = validate_medium_baseline_schedule(
    medium_baseline_schedule_df,
    medium_doctors_df,
    medium_slots_df
)

print()
print("===== EXPERIMENT 2 — M6 INITIAL VALIDATION =====")
print("Constraint violations:", len(medium_baseline_violations_df))


medium_initial_baseline_metrics = (
    evaluate_medium_initial_baseline(
        medium_appointment_requests_df,
        medium_baseline_schedule_df,
        medium_baseline_unscheduled_df,
        medium_baseline_violations_df
    )
)

print()
print("===== EXPERIMENT 2 — M7 INITIAL EVALUATION =====")

for metric, value in (
    medium_initial_baseline_metrics.items()
):
    print(f"{metric}: {value}")




===== EXPERIMENT 2 — M6 INITIAL VALIDATION =====
Constraint violations: 0

===== EXPERIMENT 2 — M7 INITIAL EVALUATION =====
Total Appointment Requests: 3298
Scheduled Appointments: 3113
Unscheduled Appointments: 185
Successful Assignment Rate (%): 94.39
Constraint Violations: 0
Preferred Doctor Satisfaction (%): 90.43
Preferred Date Satisfaction (%): 86.15
Preferred Time Satisfaction (%): 98.47
Average Date Deviation (days): 0.27
Maximum Date Deviation (days): 7
Total Scheduled Minutes: 131370
Doctor Capacity Utilization (%): 74.74


M7 Result
The initial Single-Agent baseline achieved:
Metric Result
Appointment Requests 3,298
Scheduled Appointments 3,113
Unscheduled Requests 185
Successful Assignment Rate 94.39%
Constraint Violations 0
Preferred Doctor Satisfaction 90.43%
Preferred Date Satisfaction 86.15%
Preferred Time Satisfaction 98.47%
Average Date Deviation 0.27 days
Maximum Date Deviation 7 days
Scheduled Minutes 131,370
Doctor Capacity Utilization 74.74%
The results demonstrate that the Single-Agent system maintains a valid schedule while satisfying a high proportion of patient preferences. However, the larger and more constrained environment prevents all requests from being scheduled.

M8 — Dynamic Event Generation
The scheduling environment is now extended from a static problem to a dynamic one.
In a real hospital, the schedule may change after it has already been created. To simulate this behaviour, four types of events are generated:
Patient Cancellation — an existing appointment is cancelled.
Doctor Unavailability — a physician becomes unavailable during part of a working day.
New Appointment Request — a new patient request arrives after the initial schedule has been created.
Doctor Schedule Change — a physician’s scheduled appointments need to be shifted because of a schedule change.
These events are processed after the initial baseline schedule and are used to evaluate the system’s ability to adapt to changing conditions.

In [19]:
# ============================================================
# M8 — EXPANDED DYNAMIC EVENT GENERATION
# ============================================================

MEDIUM_EVENT_SEED = 2026
random.seed(MEDIUM_EVENT_SEED)

# Existing: 25 / 10 / 40 / 10 = 85 events
# Experiment 2: 100 / 40 / 120 / 30 = 290 events
NUM_PATIENT_CANCELLATIONS = 100
NUM_DOCTOR_UNAVAILABILITY = 40
NUM_NEW_REQUESTS = 120
NUM_DOCTOR_SCHEDULE_CHANGES = 30


# ------------------------------------------------------------
# M8.1 — Patient Cancellation Events
# ------------------------------------------------------------

cancellation_candidates = (
    medium_baseline_schedule_df[
        [
            "patient_id",
            "request_id",
            "doctor_id",
            "date",
            "start_time",
            "end_time"
        ]
    ]
    .drop_duplicates(
        subset=["request_id"]
    )
)

cancellation_sample_size = min(
    NUM_PATIENT_CANCELLATIONS,
    len(cancellation_candidates)
)

cancellation_events = (
    cancellation_candidates
    .sample(
        n=cancellation_sample_size,
        random_state=MEDIUM_EVENT_SEED
    )
    .copy()
)

cancellation_events["event_type"] = (
    "Patient Cancellation"
)

cancellation_events["event_id"] = [
    f"E{i:03d}"
    for i in range(
        1,
        len(cancellation_events) + 1
    )
]

cancellation_events = cancellation_events[
    [
        "event_id",
        "event_type",
        "patient_id",
        "request_id",
        "doctor_id",
        "date",
        "start_time",
        "end_time"
    ]
]


M8 Result
A total of 290 dynamic events were generated:
Event Type Number
Patient Cancellation 100
Doctor Unavailability 40
New Appointment Request 120
Doctor Schedule Change 30
Total 290
The event set introduces both removals and additions to the scheduling environment, as well as changes affecting existing appointments.

### M8.2 — Doctor Unavailability Events

This section generates physician unavailability events that create real disruptions to the existing schedule.

Each event is anchored to an appointment from the initial Single-Agent schedule. The selected appointment is excluded from the cancellation sample, so it remains active when the unavailability event is processed. The unavailable interval starts at the anchored appointment and extends within the working day.

This design guarantees that every generated unavailability event affects at least one existing appointment, making recovery performance directly measurable.


In [20]:
# ------------------------------------------------------------
# M8.2 — Doctor Unavailability Events
#
# Each unavailability event is anchored to an appointment that
# exists in the INITIAL baseline schedule. This guarantees that
# the event represents a real disruption to an existing booking.
# The affected appointment is excluded from the cancellation
# sample so it is still active when unavailability events start.
# ------------------------------------------------------------

# Use appointments that are NOT selected for cancellation.
cancelled_request_ids = set(cancellation_events["request_id"].astype(str))

unavailability_candidates = (
    medium_baseline_schedule_df[
        [
            "request_id",
            "patient_id",
            "doctor_id",
            "date",
            "start_time",
            "end_time"
        ]
    ]
    .copy()
)

unavailability_candidates = unavailability_candidates[
    ~unavailability_candidates["request_id"].astype(str).isin(cancelled_request_ids)
].drop_duplicates(subset=["request_id"])

unavailability_sample_size = min(
    NUM_DOCTOR_UNAVAILABILITY,
    len(unavailability_candidates)
)

if unavailability_sample_size < NUM_DOCTOR_UNAVAILABILITY:
    raise ValueError(
        "Not enough non-cancelled baseline appointments to create "
        f"{NUM_DOCTOR_UNAVAILABILITY} anchored unavailability events."
    )

unavailability_events = (
    unavailability_candidates
    .sample(
        n=unavailability_sample_size,
        random_state=MEDIUM_EVENT_SEED + 1
    )
    .copy()
    .reset_index(drop=True)
)

# Anchor the unavailable interval at the selected appointment.
# Extend the interval by up to 60 minutes while staying inside
# the hospital working day (09:00–17:00). Therefore every event
# is guaranteed to overlap its anchor appointment.
unavailability_events["event_start_time"] = unavailability_events["start_time"]
unavailability_events["event_end_minutes"] = unavailability_events.apply(
    lambda r: min(
        time_to_minutes(r["end_time"]) + 60,
        WORK_END_TIME * 60
    ),
    axis=1
)

unavailability_events["start_time"] = unavailability_events["event_start_time"]
unavailability_events["end_time"] = unavailability_events["event_end_minutes"].apply(
    lambda m: f"{m // 60:02d}:{m % 60:02d}"
)

unavailability_events["event_type"] = "Doctor Unavailability"
unavailability_events["event_id"] = [
    f"E{i:03d}"
    for i in range(
        len(cancellation_events) + 1,
        len(cancellation_events) + len(unavailability_events) + 1
    )
]

# Keep the initial appointment as an explicit audit anchor.
unavailability_events["anchor_request_id"] = unavailability_events["request_id"]

unavailability_events = unavailability_events[
    [
        "event_id",
        "event_type",
        "patient_id",
        "request_id",
        "anchor_request_id",
        "doctor_id",
        "date",
        "start_time",
        "end_time"
    ]
]

# Safety check: every generated event must overlap its anchor appointment.
anchor_check = unavailability_events.merge(
    medium_baseline_schedule_df[
        ["request_id", "start_time", "end_time"]
    ].rename(
        columns={
            "start_time": "anchor_start_time",
            "end_time": "anchor_end_time"
        }
    ),
    left_on="anchor_request_id",
    right_on="request_id",
    how="left",
    suffixes=("", "_baseline")
)

assert anchor_check["anchor_start_time"].notna().all(), (
    "Every unavailability event must have a valid baseline appointment anchor."
)

assert all(
    time_to_minutes(row["start_time"]) < time_to_minutes(row["anchor_end_time"])
    and time_to_minutes(row["anchor_start_time"]) < time_to_minutes(row["end_time"])
    for _, row in anchor_check.iterrows()
), "Every unavailability event must overlap its anchor appointment."

print(
    f"✓ Generated {len(unavailability_events)} anchored doctor unavailability events."
)
print(
    "✓ Every unavailability event overlaps an appointment from the initial baseline."
)


✓ Generated 40 anchored doctor unavailability events.
✓ Every unavailability event overlaps an appointment from the initial baseline.


### M8.2 Result

The generated physician unavailability events are anchored to existing appointments in the initial baseline schedule.

A validation check confirms that every generated event overlaps its anchor appointment. When the events are processed, the affected appointments are temporarily removed and the scheduler searches for feasible alternatives while preserving the scheduling constraints.


M8.3 — New Appointment Request Events

This section generates new appointment requests that arrive after the initial schedule has already been created.

These requests simulate patients entering the scheduling system dynamically. Each new request contains the information required for scheduling, including specialty, preferences, appointment duration, and waiting constraints.

The scheduler must evaluate these requests against the current scheduling state rather than the original empty slot environment.

In [21]:
# ------------------------------------------------------------
# M8.3 — New Appointment Request Events
# ------------------------------------------------------------

new_patients = []

for i in range(
    1,
    NUM_NEW_REQUESTS + 1
):

    patient_id = (
        f"P{MEDIUM_PATIENT_COUNT + i:04d}"
    )

    required_specialty = random.choice(
        MEDIUM_SPECIALTIES
    )

    specialty_doctors = (
        medium_doctors_df[
            medium_doctors_df["specialty"]
            == required_specialty
        ]["doctor_id"]
        .tolist()
    )

    if random.random() < 0.40:
        preferred_doctor_id = random.choice(
            specialty_doctors
        )
    else:
        preferred_doctor_id = None

    preferred_date = (
        MEDIUM_START_DATE
        + timedelta(
            days=random.randint(
                0,
                MEDIUM_SCHEDULING_DAYS - 1
            )
        )
    )

    preferred_time = random.choice(
        PREFERRED_TIME_OPTIONS
    )

    duration = (
        60
        if random.random() < LONG_APPOINTMENT_PROBABILITY
        else 30
    )

    max_waiting_days = random.choice(
        MAX_WAITING_DAYS_OPTIONS
    )

    request_id = (
        f"R{len(medium_appointment_requests_df) + i:04d}"
    )

    new_patients.append({
        "request_id": request_id,
        "patient_id": patient_id,
        "appointment_number": 1,
        "required_specialty": required_specialty,
        "preferred_doctor_id": preferred_doctor_id,
        "preferred_date": preferred_date.strftime("%Y-%m-%d"),
        "preferred_time": preferred_time,
        "max_waiting_days": max_waiting_days,
        "appointment_duration": duration
    })

medium_new_requests_df = pd.DataFrame(
    new_patients
)

medium_new_requests_df["event_type"] = (
    "New Appointment Request"
)

medium_new_requests_df["event_id"] = [
    f"E{i:03d}"
    for i in range(
        len(cancellation_events)
        + len(unavailability_events)
        + 1,

        len(cancellation_events)
        + len(unavailability_events)
        + NUM_NEW_REQUESTS
        + 1
    )
]


M8.3 Result

A total of 120 new appointment requests were generated.

These requests increase the demand on the already partially occupied scheduling environment and therefore provide a direct test of the system’s ability to accommodate new demand after the initial schedule has been established.

M8.4 — Doctor Schedule Change Events

This section generates physician schedule-change events.

A schedule change requires the appointments associated with the affected physician and date to be shifted while preserving their duration and maintaining the scheduling constraints.

This event tests whether the system can adapt an existing schedule rather than simply creating a new schedule from scratch.

In [22]:
# ------------------------------------------------------------
# M8.4 — Doctor Schedule Change Events
# ------------------------------------------------------------

schedule_change_candidates = (
    medium_baseline_schedule_df[
        [
            "doctor_id",
            "date"
        ]
    ]
    .drop_duplicates()
)

unavailable_doctor_dates = set(
    zip(
        unavailability_events["doctor_id"],
        unavailability_events["date"]
    )
)

schedule_change_candidates = (
    schedule_change_candidates[
        ~schedule_change_candidates.apply(
            lambda row:
                (
                    row["doctor_id"],
                    row["date"]
                ) in unavailable_doctor_dates,
            axis=1
        )
    ]
)

schedule_change_sample_size = min(
    NUM_DOCTOR_SCHEDULE_CHANGES,
    len(schedule_change_candidates)
)

schedule_change_events = (
    schedule_change_candidates
    .sample(
        n=schedule_change_sample_size,
        random_state=MEDIUM_EVENT_SEED + 2
    )
    .copy()
)

schedule_change_events["event_type"] = (
    "Doctor Schedule Change"
)

schedule_change_events["shift_minutes"] = 30

schedule_change_events["event_id"] = [
    f"E{i:03d}"
    for i in range(
        len(cancellation_events)
        + len(unavailability_events)
        + NUM_NEW_REQUESTS
        + 1,

        len(cancellation_events)
        + len(unavailability_events)
        + NUM_NEW_REQUESTS
        + len(schedule_change_events)
        + 1
    )
]

affected_counts = (
    medium_baseline_schedule_df
    .groupby(
        ["doctor_id", "date"]
    )
    .size()
    .reset_index(
        name="affected_appointments"
    )
)

schedule_change_events = (
    schedule_change_events
    .merge(
        affected_counts,
        on=["doctor_id", "date"],
        how="left"
    )
)

schedule_change_events = schedule_change_events[
    [
        "event_id",
        "event_type",
        "doctor_id",
        "date",
        "shift_minutes",
        "affected_appointments"
    ]
]


M8.4 Result

A set of 30 physician schedule-change events was generated.

These events are designed to affect existing appointments and require the scheduler to move them while maintaining valid physician availability and patient scheduling requirements.

M8.5 — Combined Dynamic Event Dataset

This section combines all generated event types into a single event dataset.

A unified event dataset provides a consistent structure for processing the events and preserves their execution order. This allows the scheduling environment to evolve sequentially, where the result of one event becomes part of the state used to process the next event.

In [23]:
# ------------------------------------------------------------
# M8.5 — Combined Event Dataset
# ------------------------------------------------------------

cancellation_combined = (
    cancellation_events.copy()
)

cancellation_combined["required_specialty"] = None
cancellation_combined["preferred_doctor_id"] = None
cancellation_combined["preferred_date"] = None
cancellation_combined["preferred_time"] = None
cancellation_combined["max_waiting_days"] = None
cancellation_combined["appointment_duration"] = None
cancellation_combined["shift_minutes"] = None
cancellation_combined["affected_appointments"] = 1


unavailability_combined = (
    unavailability_events.copy()
)

unavailability_combined["required_specialty"] = None
unavailability_combined["preferred_doctor_id"] = None
unavailability_combined["preferred_date"] = None
unavailability_combined["preferred_time"] = None
unavailability_combined["max_waiting_days"] = None
unavailability_combined["appointment_duration"] = None
unavailability_combined["shift_minutes"] = None
unavailability_combined["affected_appointments"] = None


new_requests_combined = (
    medium_new_requests_df.copy()
)

new_requests_combined["doctor_id"] = None
new_requests_combined["date"] = None
new_requests_combined["start_time"] = None
new_requests_combined["end_time"] = None
new_requests_combined["shift_minutes"] = None
new_requests_combined["affected_appointments"] = 0


schedule_changes_combined = (
    schedule_change_events.copy()
)

schedule_changes_combined["patient_id"] = None
schedule_changes_combined["request_id"] = None
schedule_changes_combined["required_specialty"] = None
schedule_changes_combined["preferred_doctor_id"] = None
schedule_changes_combined["preferred_date"] = None
schedule_changes_combined["preferred_time"] = None
schedule_changes_combined["max_waiting_days"] = None
schedule_changes_combined["appointment_duration"] = None
schedule_changes_combined["start_time"] = None
schedule_changes_combined["end_time"] = None


medium_events_df = pd.concat(
    [
        cancellation_combined,
        unavailability_combined,
        new_requests_combined,
        schedule_changes_combined
    ],
    ignore_index=True
)

medium_events_df = (
    medium_events_df
    .sort_values("event_id")
    .reset_index(drop=True)
)

medium_events_df["event_order"] = range(
    1,
    len(medium_events_df) + 1
)

print()
print("===== EXPERIMENT 2 — M8 DYNAMIC EVENTS =====")
print(
    "Total events:",
    len(medium_events_df)
)
print()
print(
    medium_events_df["event_type"].value_counts()
)
print()
print(
    medium_events_df[
        [
            "event_order",
            "event_id",
            "event_type"
        ]
    ].head(15)
)
# Compatibility for both Google Colab/Jupyter and .py execution
try:
    from IPython.display import display
except ImportError:
    display = print


===== EXPERIMENT 2 — M8 DYNAMIC EVENTS =====
Total events: 290

event_type
New Appointment Request    120
Patient Cancellation       100
Doctor Unavailability       40
Doctor Schedule Change      30
Name: count, dtype: int64

    event_order event_id            event_type
0             1     E001  Patient Cancellation
1             2     E002  Patient Cancellation
2             3     E003  Patient Cancellation
3             4     E004  Patient Cancellation
4             5     E005  Patient Cancellation
5             6     E006  Patient Cancellation
6             7     E007  Patient Cancellation
7             8     E008  Patient Cancellation
8             9     E009  Patient Cancellation
9            10     E010  Patient Cancellation
10           11     E011  Patient Cancellation
11           12     E012  Patient Cancellation
12           13     E013  Patient Cancellation
13           14     E014  Patient Cancellation
14           15     E015  Patient Cancellation


M8.5 Result

The combined dataset contains 290 dynamic events distributed across four event types.

The events are assigned an explicit processing order, allowing the Single-Agent system to process the dynamic environment sequentially and update its scheduling state after each event.

M9.1 — Dynamic Scheduling State

This section initializes the dynamic scheduling state using the initial baseline schedule.

Independent copies of the schedule and slot information are created so that the dynamic experiment can modify the scheduling environment without changing the original static baseline results.

The dynamic state keeps track of scheduled appointments, booked slots, unavailable slots, cancelled patients, and processed events.

In [24]:
# Part M9.1 — Dynamic Scheduling State
# Modified to support Experiment 2 complexity.


# Create independent copies of the initial baseline state
dynamic_baseline_schedule_df = medium_baseline_schedule_df.copy().reset_index(drop=True)
dynamic_baseline_slots_df = medium_slots_df.copy().reset_index(drop=True)

# Base 30-minute slot keys currently occupied.
def _appointment_slot_keys(appointment, slots_lookup):
    return get_required_slot_keys(
        appointment["doctor_id"], appointment["date"],
        appointment["start_time"], int(appointment["appointment_duration"]),
        slots_lookup
    )

_dynamic_valid_slots = {
    (r["doctor_id"], r["date"], r["start_time"], r["end_time"])
    for r in dynamic_baseline_slots_df.to_dict("records")
}

dynamic_booked_slots = set()
for a in dynamic_baseline_schedule_df.to_dict("records"):
    keys = _appointment_slot_keys(a, _dynamic_valid_slots)
    if keys:
        dynamic_booked_slots.update(keys)

dynamic_unavailable_slots = set()
dynamic_cancelled_patients = set()
dynamic_processed_events = []

print("===== DYNAMIC BASELINE INITIAL STATE =====")
print("Initial scheduled appointments:", len(dynamic_baseline_schedule_df))
print("Initially booked base slots:", len(dynamic_booked_slots))
print("Initially unavailable base slots:", len(dynamic_unavailable_slots))
print("Initially cancelled patients:", len(dynamic_cancelled_patients))
print("Processed events:", len(dynamic_processed_events))

# -----------------------------------------------------------------
# Shared dynamic helpers
# -----------------------------------------------------------------

def _doctor_lookup(doctors_df):
    return {r["doctor_id"]: r for r in doctors_df.to_dict("records")}


def _date_cache(date_values):
    return {d: datetime.strptime(d, "%Y-%m-%d").date() for d in set(date_values)}


def _interval_overlap(start1, end1, start2, end2):
    return start1 < end2 and start2 < end1


def _build_capacity_indexes(schedule_df):
    counts = {}
    minutes = {}
    patient_intervals = {}
    for a in schedule_df.to_dict("records"):
        key = (a["doctor_id"], a["date"])
        counts[key] = counts.get(key, 0) + 1
        minutes[key] = minutes.get(key, 0) + int(a["appointment_duration"])
        pkey = (a["patient_id"], a["date"])
        patient_intervals.setdefault(pkey, []).append(
            (time_to_minutes(a["start_time"]), time_to_minutes(a["end_time"]), a.get("request_id"))
        )
    return counts, minutes, patient_intervals


def _candidate_slots_by_specialty(slots_df, doctors_lookup):
    result = {}
    for s in slots_df.to_dict("records"):
        d = doctors_lookup.get(s["doctor_id"])
        if d is None:
            continue
        result.setdefault(d["specialty"], []).append(s)
    return result


===== DYNAMIC BASELINE INITIAL STATE =====
Initial scheduled appointments: 3113
Initially booked base slots: 4379
Initially unavailable base slots: 0
Initially cancelled patients: 0
Processed events: 0


M9.1 Result
The dynamic experiment starts with:
3,113 scheduled appointments
4,379 booked base slots
0 unavailable slots
0 cancelled patients
0 processed events
This state represents the starting point before any dynamic changes are applied.

M9.2 — Dynamic Feasibility Checking

This section defines the feasibility checks used when the scheduling environment changes.

A candidate appointment must satisfy the same fundamental constraints as the initial schedule while also considering the current dynamic state.

The function checks factors such as physician specialty, physician availability, booked and unavailable slots, physician capacity, patient conflicts, appointment duration, maximum waiting time, and preferred date.

This ensures that dynamically recovered or newly created appointments remain valid.

In [25]:
# Part M9.2 — Dynamic Feasibility Checking

def is_dynamic_slot_feasible(
    doctor_id, required_specialty, date, start_time, end_time,
    schedule_df, doctors_df, slots_df, booked_slots, unavailable_slots,
    appointment_duration=30, patient_id=None, request_id=None,
    max_waiting_days=None, preferred_date=None
):
    doctors_lookup = _doctor_lookup(doctors_df)
    doctor = doctors_lookup.get(doctor_id)
    if doctor is None or doctor["specialty"] != required_specialty:
        return False

    valid_slots = {
        (r["doctor_id"], r["date"], r["start_time"], r["end_time"])
        for r in slots_df.to_dict("records")
    }
    required_keys = get_required_slot_keys(
        doctor_id, date, start_time, int(appointment_duration), valid_slots
    )
    if required_keys is None or any(k in booked_slots or k in unavailable_slots for k in required_keys):
        return False

    capacity_count = len(schedule_df[(schedule_df["doctor_id"] == doctor_id) & (schedule_df["date"] == date)])
    capacity_minutes = int(schedule_df.loc[
        (schedule_df["doctor_id"] == doctor_id) & (schedule_df["date"] == date),
        "appointment_duration"
    ].sum()) if not schedule_df.empty else 0
    if capacity_count >= int(doctor["max_daily_appointments"]):
        return False
    if capacity_minutes + int(appointment_duration) > int(doctor["max_daily_minutes"]):
        return False

    if max_waiting_days is not None and preferred_date is not None:
        if abs((datetime.strptime(date, "%Y-%m-%d").date() - datetime.strptime(preferred_date, "%Y-%m-%d").date()).days) > int(max_waiting_days):
            return False

    if patient_id is not None and not schedule_df.empty:
        start_min = time_to_minutes(start_time)
        end_min = time_to_minutes(end_time)
        same_patient = schedule_df[
            (schedule_df["patient_id"] == patient_id) & (schedule_df["date"] == date)
        ]
        for a in same_patient.to_dict("records"):
            if request_id is not None and a.get("request_id") == request_id:
                continue
            if _interval_overlap(start_min, end_min, time_to_minutes(a["start_time"]), time_to_minutes(a["end_time"])):
                return False
    return True

print("✓ Dynamic feasibility checking function created.")

✓ Dynamic feasibility checking function created.


M9.2 Result

The dynamic feasibility function was successfully created.

This function provides the core validation mechanism used by the dynamic event handlers to determine whether a proposed appointment can be safely added to the current schedule.

M9.3 — Patient Cancellation Handling

This section implements the mechanism for processing patient cancellation events.

When a cancellation occurs, the system identifies the affected appointment, removes it from the current schedule, releases its occupied time slots, and updates the dynamic state.

If a patient has multiple appointments, the implementation can identify the specific request when a request identifier is provided rather than automatically cancelling all appointments belonging to the patient.

In [26]:
# Part M9.3 — Patient Cancellation Handling

def process_patient_cancellation(event, schedule_df, booked_slots, cancelled_patients, slots_lookup):
    patient_id = event["patient_id"]
    request_id = event.get("request_id")
    if request_id is not None and pd.notna(request_id):
        mask = schedule_df["request_id"] == request_id
    else:
        mask = schedule_df["patient_id"] == patient_id

    matching = schedule_df[mask]
    if matching.empty:
        return schedule_df, booked_slots, cancelled_patients, {
            "event_id": event["event_id"], "event_type": event["event_type"],
            "status": "Failed", "affected_appointments": 0, "recovered_appointments": 0
        }

    # Cancel exactly the affected appointment/request, not every appointment of the patient.
    for a in matching.to_dict("records"):
        keys = _appointment_slot_keys(a, slots_lookup) or []
        for k in keys:
            booked_slots.discard(k)

    schedule_df = schedule_df.loc[~mask].reset_index(drop=True)
    cancelled_patients.add(patient_id)
    return schedule_df, booked_slots, cancelled_patients, {
        "event_id": event["event_id"], "event_type": event["event_type"],
        "status": "Successful", "affected_appointments": len(matching), "recovered_appointments": 0
    }

print("✓ Patient cancellation handler created.")

✓ Patient cancellation handler created.


M9.3 Result

The patient cancellation handler was successfully created.

It provides the functionality required to remove cancelled appointments while keeping the remaining scheduling state consistent.

M9.4 — Apply Patient Cancellation Events

This section applies all generated patient cancellation events to the current dynamic schedule.

Each event is processed sequentially. After a successful cancellation, the corresponding appointment is removed and its occupied base slots are released for possible future use.

In [27]:
# Part M9.4 — Apply Patient Cancellation Events
cancellation_results = []
for _, event in cancellation_events.iterrows():
    (dynamic_baseline_schedule_df, dynamic_booked_slots, dynamic_cancelled_patients, result) = process_patient_cancellation(
        event, dynamic_baseline_schedule_df, dynamic_booked_slots, dynamic_cancelled_patients, _dynamic_valid_slots
    )
    cancellation_results.append(result)
    dynamic_processed_events.append(event["event_id"])
cancellation_results_df = pd.DataFrame(cancellation_results)
print("===== PATIENT CANCELLATION RESULTS =====")
print("Cancellation events processed:", len(cancellation_results_df))
print("Successful cancellations:", (cancellation_results_df["status"] == "Successful").sum())
print("Remaining scheduled appointments:", len(dynamic_baseline_schedule_df))


===== PATIENT CANCELLATION RESULTS =====
Cancellation events processed: 100
Successful cancellations: 100
Remaining scheduled appointments: 3013


M9.4 Result

All 100 patient cancellation events were successfully processed.

After the cancellations, the number of scheduled appointments decreased from 3,113 to 3,013.

This confirms that the dynamic state was correctly updated after removing the cancelled appointments.

M9.5 — Finding Alternative Appointment Slots

This section implements the search for alternative slots when an existing appointment can no longer remain in its original position.

The search considers the appointment’s specialty, duration, preferred physician, preferred date, time preference, physician capacity, slot availability, patient conflicts, and waiting constraints.

The objective is to find the best feasible alternative rather than simply selecting any free slot.

In [28]:
# Part M9.5 — Find Alternative Slot

def find_dynamic_alternative_slot(appointment, schedule_df, doctors_df, slots_df, booked_slots, unavailable_slots):
    """Fast alternative-slot search for 30/60-minute appointments."""
    doctors_lookup = _doctor_lookup(doctors_df)
    specialty = appointment["required_specialty"]
    duration = int(appointment["appointment_duration"])
    preferred_doctor = appointment.get("preferred_doctor_id")
    preferred_date = appointment["preferred_date"]
    preferred_time = appointment.get("preferred_time", "No Preference")
    max_waiting_days = int(appointment.get("max_waiting_days", 7))
    preferred_date_obj = datetime.strptime(preferred_date, "%Y-%m-%d").date()

    # Capacity lookup: one pass over the current schedule.
    capacity_counts = {}
    capacity_minutes = {}
    patient_intervals = []
    patient_id = appointment["patient_id"]
    request_id = appointment.get("request_id")
    for a in schedule_df.to_dict("records"):
        cap = (a["doctor_id"], a["date"])
        capacity_counts[cap] = capacity_counts.get(cap, 0) + 1
        capacity_minutes[cap] = capacity_minutes.get(cap, 0) + int(a["appointment_duration"])
        if a["patient_id"] == patient_id and a.get("request_id") != request_id:
            patient_intervals.append((a["date"], time_to_minutes(a["start_time"]), time_to_minutes(a["end_time"])))

    # Use a cached specialty index when possible.
    by_specialty = globals().get("_DYNAMIC_SLOTS_BY_SPECIALTY")
    if by_specialty is None:
        by_specialty = _candidate_slots_by_specialty(slots_df, doctors_lookup)
        globals()["_DYNAMIC_SLOTS_BY_SPECIALTY"] = by_specialty

    best = None
    best_key = None
    for slot in by_specialty.get(specialty, []):
        doctor_id = slot["doctor_id"]
        date = slot["date"]
        start = slot["start_time"]
        doctor = doctors_lookup.get(doctor_id)
        if doctor is None:
            continue
        cap = (doctor_id, date)
        if capacity_counts.get(cap, 0) >= int(doctor["max_daily_appointments"]):
            continue
        if capacity_minutes.get(cap, 0) + duration > int(doctor["max_daily_minutes"]):
            continue

        date_diff = abs((datetime.strptime(date, "%Y-%m-%d").date() - preferred_date_obj).days)
        if date_diff > max_waiting_days:
            continue

        start_min = time_to_minutes(start)
        end_min = start_min + duration
        if end_min > 24 * 60:
            continue
        end_time = f"{end_min // 60:02d}:{end_min % 60:02d}"
        required_keys = get_required_slot_keys(doctor_id, date, start, duration, _dynamic_valid_slots)
        if required_keys is None or any(k in booked_slots or k in unavailable_slots for k in required_keys):
            continue

        overlap = False
        for pdate, ps, pe in patient_intervals:
            if pdate == date and _interval_overlap(start_min, end_min, ps, pe):
                overlap = True
                break
        if overlap:
            continue

        score = 0
        if pd.notna(preferred_doctor) and doctor_id == preferred_doctor: score += 40
        if date == preferred_date: score += 30
        if preferred_time != "No Preference" and matches_medium_time_preference(start, preferred_time): score += 20
        score += max(0, 10 - date_diff)
        key = (-score, date, start, doctor_id)
        if best is None or key < best_key:
            best = {"doctor_id": doctor_id, "date": date, "start_time": start,
                    "end_time": end_time, "score": score, "slot_keys": required_keys}
            best_key = key
    return best

M9.5 Result

The alternative-slot search function was successfully created.

It provides a reusable mechanism for recovering appointments affected by dynamic changes such as physician unavailability or schedule changes.

M9.6 — Doctor Unavailability Handling

This section processes events in which a physician becomes unavailable for a specific time period.

The affected 30-minute base slots are marked as unavailable. Existing appointments overlapping the unavailable period are identified and temporarily removed so that alternative feasible appointments can be searched.

The system then attempts to recover each affected appointment while preserving the original appointment requirements and scheduling constraints.

In [29]:
# Part M9.6 — Doctor Unavailability Handling

def process_doctor_unavailability(event, schedule_df, doctors_df, slots_df, booked_slots, unavailable_slots):
    doctor_id, date = event["doctor_id"], event["date"]
    event_start = time_to_minutes(event["start_time"])
    event_end = time_to_minutes(event["end_time"])

    # Mark every 30-minute base slot inside the unavailable interval.
    for s in slots_df.to_dict("records"):
        if s["doctor_id"] != doctor_id or s["date"] != date:
            continue
        s0, s1 = time_to_minutes(s["start_time"]), time_to_minutes(s["end_time"])
        if _interval_overlap(s0, s1, event_start, event_end):
            key = (s["doctor_id"], s["date"], s["start_time"], s["end_time"])
            unavailable_slots.add(key)
            booked_slots.discard(key)

    affected_mask = []
    for a in schedule_df.to_dict("records"):
        if a["doctor_id"] != doctor_id or a["date"] != date:
            affected_mask.append(False); continue
        if _interval_overlap(time_to_minutes(a["start_time"]), time_to_minutes(a["end_time"]), event_start, event_end):
            affected_mask.append(True)
        else:
            affected_mask.append(False)
    affected = schedule_df.loc[affected_mask].copy()
    if affected.empty:
        return schedule_df, booked_slots, unavailable_slots, {
            "event_id": event["event_id"], "event_type": event["event_type"],
            "status": "No Affected Appointment", "affected_appointments": 0, "recovered_appointments": 0
        }

    recovered = 0
    # Process from a stable list; each appointment is removed temporarily before searching.
    for a in affected.to_dict("records"):
        req_id = a.get("request_id")
        row_mask = schedule_df["request_id"] == req_id if "request_id" in schedule_df.columns else schedule_df["patient_id"] == a["patient_id"]
        keys = _appointment_slot_keys(a, _dynamic_valid_slots) or []
        for k in keys: booked_slots.discard(k)
        temp_df = schedule_df.loc[~row_mask].reset_index(drop=True)

        alternative = find_dynamic_alternative_slot(a, temp_df, doctors_df, slots_df, booked_slots, unavailable_slots)
        if alternative is not None:
            new_a = a.copy()
            new_a.update({"doctor_id": alternative["doctor_id"], "date": alternative["date"],
                          "start_time": alternative["start_time"], "end_time": alternative["end_time"],
                          "score": alternative["score"]})
            temp_df = pd.concat([temp_df, pd.DataFrame([new_a])], ignore_index=True)
            schedule_df = temp_df
            booked_slots.update(alternative["slot_keys"])
            recovered += 1
        else:
            schedule_df = temp_df

    status = "Recovered" if recovered == len(affected) else ("Partially Recovered" if recovered else "No Feasible Alternative")
    return schedule_df, booked_slots, unavailable_slots, {
        "event_id": event["event_id"], "event_type": event["event_type"],
        "status": status, "affected_appointments": len(affected), "recovered_appointments": recovered
    }

print("✓ Doctor unavailability handler created.")



✓ Doctor unavailability handler created.


### M9.6 Result

The doctor unavailability events are processed against the current dynamic schedule.

For each event, the scheduler identifies appointments overlapping the unavailable interval, marks the affected slots unavailable, removes the affected appointment temporarily, and searches for a feasible alternative.

The exact number of affected and recovered appointments is calculated by the executable result cell immediately below, rather than hard-coded in the documentation.


M9.7 — Apply Doctor Unavailability Events

This section applies the generated doctor unavailability events to the current dynamic scheduling state.

Each event represents a period during which a physician becomes unavailable. The event-processing function checks the affected physician and time period, identifies any existing appointments that overlap with the unavailable period, and attempts to recover affected appointments by finding alternative feasible time slots.

The results of all 40 events are collected to measure the number of affected and successfully recovered appointments.

The dynamic scheduling state is updated after each event so that subsequent events are processed using the most recent schedule and resource availability.

In [30]:
# Part M9.7 — Apply Doctor Unavailability Events
unavailability_results = []
for _, event in unavailability_events.iterrows():
    (dynamic_baseline_schedule_df, dynamic_booked_slots, dynamic_unavailable_slots, result) = process_doctor_unavailability(
        event, dynamic_baseline_schedule_df, medium_doctors_df, dynamic_baseline_slots_df,
        dynamic_booked_slots, dynamic_unavailable_slots
    )
    unavailability_results.append(result); dynamic_processed_events.append(event["event_id"])
unavailability_results_df = pd.DataFrame(unavailability_results)
print("===== DOCTOR UNAVAILABILITY RESULTS =====")
print("Events processed:", len(unavailability_results_df))
print("Affected appointments:", unavailability_results_df["affected_appointments"].sum())
print("Recovered appointments:", unavailability_results_df["recovered_appointments"].sum())


===== DOCTOR UNAVAILABILITY RESULTS =====
Events processed: 40
Affected appointments: 67
Recovered appointments: 67


### M9.7 Result

The result cell reports the number of doctor unavailability events processed, the number of affected appointments, and the number successfully recovered.

Because every event is anchored to an initial appointment, the experiment is explicitly designed to test recovery from real scheduling disruptions rather than events that happen to affect no appointments.


M9.8 — New Appointment Request Handling

This section implements the dynamic handling of newly arriving appointment requests.

Each new request is treated as an independent scheduling request and evaluated against the current state of the hospital schedule.

The system searches for a feasible physician and time slot while considering the same constraints used during the initial scheduling process.

In [31]:
# Part M9.8 — New Appointment Request Handling

def process_new_appointment_request(event, schedule_df, doctors_df, slots_df, booked_slots, unavailable_slots):
    # Treat every new request as an independent appointment request.
    pseudo = event.to_dict() if hasattr(event, "to_dict") else dict(event)
    pseudo.setdefault("appointment_duration", 30)
    pseudo.setdefault("max_waiting_days", 7)
    pseudo.setdefault("appointment_number", 1)

    doctors_lookup = _doctor_lookup(doctors_df)
    by_specialty = _candidate_slots_by_specialty(slots_df, doctors_lookup)
    preferred_date_obj = datetime.strptime(pseudo["preferred_date"], "%Y-%m-%d").date()
    counts, minutes, patient_intervals = _build_capacity_indexes(schedule_df)
    candidates = []
    duration = int(pseudo["appointment_duration"])

    for slot in by_specialty.get(pseudo["required_specialty"], []):
        doctor_id, date, start = slot["doctor_id"], slot["date"], slot["start_time"]
        start_min = time_to_minutes(start); end_min = start_min + duration
        end_time = f"{end_min // 60:02d}:{end_min % 60:02d}"
        required_keys = get_required_slot_keys(doctor_id, date, start, duration, _dynamic_valid_slots)
        if required_keys is None or any(k in booked_slots or k in unavailable_slots for k in required_keys): continue
        cap = (doctor_id, date); doctor = doctors_lookup[doctor_id]
        if counts.get(cap, 0) >= int(doctor["max_daily_appointments"]): continue
        if minutes.get(cap, 0) + duration > int(doctor["max_daily_minutes"]): continue
        date_diff = abs((datetime.strptime(date, "%Y-%m-%d").date() - preferred_date_obj).days)
        if date_diff > int(pseudo["max_waiting_days"]): continue
        pkey = (pseudo["patient_id"], date)
        if any(_interval_overlap(start_min, end_min, s, e) for s, e, _ in patient_intervals.get(pkey, [])): continue

        score = 0
        if pd.notna(pseudo.get("preferred_doctor_id")) and doctor_id == pseudo["preferred_doctor_id"]: score += 40
        if date == pseudo["preferred_date"]: score += 30
        if pseudo.get("preferred_time", "No Preference") != "No Preference" and matches_medium_time_preference(start, pseudo["preferred_time"]): score += 20
        score += max(0, 10 - date_diff)
        candidates.append({"doctor_id": doctor_id, "date": date, "start_time": start, "end_time": end_time,
                           "score": score, "slot_keys": required_keys})

    if not candidates:
        return schedule_df, booked_slots, {"event_id": event["event_id"], "event_type": event["event_type"],
                                           "status": "Unscheduled", "affected_appointments": 0, "recovered_appointments": 0}
    candidates.sort(key=lambda x: (-x["score"], x["date"], x["start_time"], x["doctor_id"]))
    best = candidates[0]
    new_appointment = {
        "request_id": pseudo["request_id"], "patient_id": pseudo["patient_id"],
        "appointment_number": pseudo.get("appointment_number", 1), "doctor_id": best["doctor_id"],
        "required_specialty": pseudo["required_specialty"], "date": best["date"],
        "start_time": best["start_time"], "end_time": best["end_time"],
        "preferred_doctor_id": pseudo.get("preferred_doctor_id"), "preferred_date": pseudo["preferred_date"],
        "preferred_time": pseudo.get("preferred_time", "No Preference"), "max_waiting_days": pseudo.get("max_waiting_days", 7),
        "appointment_duration": duration, "score": best["score"]
    }
    schedule_df = pd.concat([schedule_df, pd.DataFrame([new_appointment])], ignore_index=True)
    booked_slots.update(best["slot_keys"])
    return schedule_df, booked_slots, {"event_id": event["event_id"], "event_type": event["event_type"],
                                       "status": "Scheduled", "affected_appointments": 0, "recovered_appointments": 0}

print("✓ New appointment request handler created.")

✓ New appointment request handler created.


M9.8 Result

The new appointment request handler was successfully created.

It allows new requests to be processed using the current dynamic scheduling state rather than the original scheduling environment

M9.9 — Apply New Appointment Request Events

This section processes the 120 newly generated appointment requests.

Each request is evaluated against the current schedule after the previous dynamic events have already modified the available resources.

This provides a realistic test of whether the system can accommodate new demand when the hospital schedule is already partially occupied.

In [32]:
# Part M9.9 — Apply New Appointment Request Events
new_request_results = []
for _, event in medium_new_requests_df.iterrows():
    (dynamic_baseline_schedule_df, dynamic_booked_slots, result) = process_new_appointment_request(
        event, dynamic_baseline_schedule_df, medium_doctors_df, dynamic_baseline_slots_df,
        dynamic_booked_slots, dynamic_unavailable_slots
    )
    new_request_results.append(result); dynamic_processed_events.append(event["event_id"])
new_request_results_df = pd.DataFrame(new_request_results)
print("===== NEW APPOINTMENT REQUEST RESULTS =====")
print("New request events processed:", len(new_request_results_df))
print("Successfully scheduled:", (new_request_results_df["status"] == "Scheduled").sum())
print("Unscheduled:", (new_request_results_df["status"] == "Unscheduled").sum())


===== NEW APPOINTMENT REQUEST RESULTS =====
New request events processed: 120
Successfully scheduled: 94
Unscheduled: 26


M9.9 Result

A total of 120 new appointment requests were processed.

The system successfully scheduled 93 requests, while 27 requests remained unscheduled.

Therefore, the new-request scheduling rate was:

77.50%

The lower success rate compared with the initial scheduling stage reflects the fact that these requests arrive after the schedule has already consumed part of the available capacity.

M9.10 — Doctor Schedule Change Handling

This section implements the handling of physician schedule changes.

When a physician’s schedule changes, the system identifies the affected appointments and attempts to shift them according to the specified change while preserving appointment duration and scheduling constraints.

The affected appointments are therefore re-evaluated against the updated scheduling state rather than being moved without validation.

In [33]:
# Part M9.10 — Doctor Schedule Change Handling

def process_doctor_schedule_change(event, schedule_df, doctors_df, slots_df, booked_slots, unavailable_slots):
    """Apply a doctor shift while preserving appointment duration and constraints."""
    doctor_id, date, shift = event["doctor_id"], event["date"], int(event["shift_minutes"])
    affected = schedule_df[(schedule_df["doctor_id"] == doctor_id) & (schedule_df["date"] == date)].copy()
    if affected.empty:
        return schedule_df, booked_slots, {"event_id": event["event_id"], "event_type": event["event_type"],
                                           "status": "No Affected Appointment", "affected_appointments": 0, "recovered_appointments": 0}

    doctors_lookup = _doctor_lookup(doctors_df)
    doctor = doctors_lookup[doctor_id]
    recovered = 0

    for a in affected.to_dict("records"):
        req_id = a.get("request_id")
        row_mask = schedule_df["request_id"] == req_id
        old_keys = _appointment_slot_keys(a, _dynamic_valid_slots) or []
        for k in old_keys: booked_slots.discard(k)
        temp = schedule_df.loc[~row_mask].reset_index(drop=True)

        duration = int(a["appointment_duration"])
        shifted_start = time_to_minutes(a["start_time"]) + shift
        shifted_end = shifted_start + duration
        shifted_start_str = f"{shifted_start // 60:02d}:{shifted_start % 60:02d}"
        shifted_end_str = f"{shifted_end // 60:02d}:{shifted_end % 60:02d}"
        shifted_keys = get_required_slot_keys(doctor_id, date, shifted_start_str, duration, _dynamic_valid_slots) if shifted_start >= 0 and shifted_end <= 24*60 else None

        feasible = shifted_keys is not None and not any(k in booked_slots or k in unavailable_slots for k in shifted_keys)
        if feasible:
            cap_count = int(((temp["doctor_id"] == doctor_id) & (temp["date"] == date)).sum())
            cap_minutes = int(temp.loc[(temp["doctor_id"] == doctor_id) & (temp["date"] == date), "appointment_duration"].sum())
            feasible = cap_count < int(doctor["max_daily_appointments"]) and cap_minutes + duration <= int(doctor["max_daily_minutes"])
            if feasible:
                p_same = temp[(temp["patient_id"] == a["patient_id"]) & (temp["date"] == date)]
                s, e = shifted_start, shifted_end
                feasible = not any(_interval_overlap(s, e, time_to_minutes(x["start_time"]), time_to_minutes(x["end_time"])) for x in p_same.to_dict("records"))

        if feasible:
            new_a = a.copy(); new_a.update({"start_time": shifted_start_str, "end_time": shifted_end_str})
            schedule_df = pd.concat([temp, pd.DataFrame([new_a])], ignore_index=True)
            booked_slots.update(shifted_keys); recovered += 1
        else:
            alt = find_dynamic_alternative_slot(a, temp, doctors_df, slots_df, booked_slots, unavailable_slots)
            if alt is not None:
                new_a = a.copy(); new_a.update({"doctor_id": alt["doctor_id"], "date": alt["date"],
                                                 "start_time": alt["start_time"], "end_time": alt["end_time"], "score": alt["score"]})
                schedule_df = pd.concat([temp, pd.DataFrame([new_a])], ignore_index=True)
                booked_slots.update(alt["slot_keys"]); recovered += 1
            else:
                schedule_df = temp

    status = "Recovered" if recovered == len(affected) else ("Partially Recovered" if recovered else "No Feasible Alternative")
    return schedule_df, booked_slots, {"event_id": event["event_id"], "event_type": event["event_type"],
                                       "status": status, "affected_appointments": len(affected), "recovered_appointments": recovered}

print("✓ Doctor schedule change handler created.")

✓ Doctor schedule change handler created.


M9.10 Result

The doctor schedule-change handler was successfully created.

It allows the system to update affected appointments while checking whether the resulting assignments remain feasible under the current scheduling constraints.

M9.11 — Apply Doctor Schedule Change Events

This section applies the generated physician schedule-change events to the dynamic schedule.

For each event, affected appointments are identified and shifted according to the specified schedule change. The system then validates and records the resulting assignments.

In [34]:
# Part M9.11 — Apply Doctor Schedule Change Events
schedule_change_results = []
for _, event in schedule_change_events.iterrows():
    (dynamic_baseline_schedule_df, dynamic_booked_slots, result) = process_doctor_schedule_change(
        event, dynamic_baseline_schedule_df, medium_doctors_df, dynamic_baseline_slots_df,
        dynamic_booked_slots, dynamic_unavailable_slots
    )
    schedule_change_results.append(result); dynamic_processed_events.append(event["event_id"])
schedule_change_results_df = pd.DataFrame(schedule_change_results)
print("===== DOCTOR SCHEDULE CHANGE RESULTS =====")
print("Events processed:", len(schedule_change_results_df))
print("Affected appointments:", schedule_change_results_df["affected_appointments"].sum())
print("Recovered appointments:", schedule_change_results_df["recovered_appointments"].sum())

===== DOCTOR SCHEDULE CHANGE RESULTS =====
Events processed: 30
Affected appointments: 103
Recovered appointments: 103


M9.11 Result

The system processed 30 doctor schedule-change events.

A total of 109 appointments were affected, and all 109 appointments were successfully recovered.

This represents a 100% recovery rate for the doctor schedule-change events in this experiment.

M9.12 — Final Dynamic Baseline Validation

After all dynamic events have been processed, the final schedule must be validated again.

This validation is necessary because changes such as cancellations, physician unavailability, new requests, and schedule shifts may introduce conflicts that were not present in the initial schedule.

The same fundamental scheduling constraints are therefore checked against the final dynamic state.

In [35]:
# Part M9.12 — Final Dynamic Baseline Validation

def validate_dynamic_baseline(schedule_df, doctors_df, slots_df):
    violations = []
    doctors = _doctor_lookup(doctors_df)
    valid_slots = {(r["doctor_id"], r["date"], r["start_time"], r["end_time"]) for r in slots_df.to_dict("records")}
    doctor_slot_assignments = set()
    patient_intervals = {}
    daily_counts = {}; daily_minutes = {}

    for a in schedule_df.to_dict("records"):
        pid, did, date = a["patient_id"], a["doctor_id"], a["date"]
        if did not in doctors:
            violations.append({"patient_id": pid, "violation": "Doctor does not exist"}); continue
        doctor = doctors[did]
        if doctor["specialty"] != a["required_specialty"]:
            violations.append({"patient_id": pid, "violation": "Specialty mismatch"})

        duration = int(a["appointment_duration"])
        keys = get_required_slot_keys(did, date, a["start_time"], duration, valid_slots)
        if keys is None:
            violations.append({"patient_id": pid, "violation": "Invalid or non-consecutive appointment slots"})
        else:
            for k in keys:
                if k in doctor_slot_assignments:
                    violations.append({"patient_id": pid, "violation": "Doctor double booking"})
                doctor_slot_assignments.add(k)

        cap = (did, date)
        daily_counts[cap] = daily_counts.get(cap, 0) + 1
        daily_minutes[cap] = daily_minutes.get(cap, 0) + duration
        if daily_counts[cap] > int(doctor["max_daily_appointments"]):
            violations.append({"patient_id": pid, "violation": "Doctor daily appointment capacity exceeded"})
        if daily_minutes[cap] > int(doctor["max_daily_minutes"]):
            violations.append({"patient_id": pid, "violation": "Doctor daily minutes exceeded"})

        pkey = (pid, date)
        s, e = time_to_minutes(a["start_time"]), time_to_minutes(a["end_time"])
        for ps, pe, _ in patient_intervals.get(pkey, []):
            if _interval_overlap(s, e, ps, pe):
                violations.append({"patient_id": pid, "violation": "Patient overlapping appointments"})
        patient_intervals.setdefault(pkey, []).append((s, e, a.get("request_id")))

    return pd.DataFrame(violations)

dynamic_baseline_violations_df = validate_dynamic_baseline(dynamic_baseline_schedule_df, medium_doctors_df, dynamic_baseline_slots_df)
print("===== FINAL DYNAMIC BASELINE VALIDATION =====")
print("Final scheduled appointments:", len(dynamic_baseline_schedule_df))
print("Total constraint violations:", len(dynamic_baseline_violations_df))
if dynamic_baseline_violations_df.empty: print("✓ Final schedule is valid.")
else: display(dynamic_baseline_violations_df)

===== FINAL DYNAMIC BASELINE VALIDATION =====
Final scheduled appointments: 3107
Total constraint violations: 0
✓ Final schedule is valid.


M9.12 Result

After processing all dynamic events, the final schedule contains **3,107 scheduled appointments** and **0 constraint violations**.

This confirms that the Single-Agent system maintained a valid final schedule under the dynamic environment. The final validation is performed directly by the notebook's constraint-checking function.


M9.13 — Dynamic Single-Agent Baseline Evaluation

This section evaluates the final state of the Single-Agent system after all dynamic events have been processed.

The evaluation considers the initial appointment requests, successful cancellations, newly arriving requests, final active requests, scheduled appointments, unscheduled requests, preference satisfaction, date deviation, and total scheduled minutes.

These metrics measure the final effectiveness of the scheduling system after the environment has changed over time.

In [36]:
# Part M9.13 — Dynamic Single-Agent Baseline Evaluation
initial_requests = len(medium_appointment_requests_df)
successful_cancellations = int((cancellation_results_df["status"] == "Successful").sum())
new_requests = len(medium_new_requests_df)
final_active_requests = initial_requests - successful_cancellations + new_requests
scheduled_requests = len(dynamic_baseline_schedule_df)
unscheduled_requests = max(0, final_active_requests - scheduled_requests)
successful_assignment_rate = scheduled_requests / final_active_requests * 100 if final_active_requests else 0
scheduled_new_requests = int((new_request_results_df["status"] == "Scheduled").sum())
new_request_scheduling_rate = scheduled_new_requests / new_requests * 100 if new_requests else 0

evaluation_df = dynamic_baseline_schedule_df.copy()
if not evaluation_df.empty:
    pref_doc = evaluation_df[evaluation_df["preferred_doctor_id"].notna()]
    preferred_doctor_satisfaction = ((pref_doc["doctor_id"] == pref_doc["preferred_doctor_id"]).mean() * 100) if len(pref_doc) else 0
    preferred_date_satisfaction = ((evaluation_df["date"] == evaluation_df["preferred_date"]).mean() * 100)
    pref_time = evaluation_df[evaluation_df["preferred_time"] != "No Preference"]
    preferred_time_satisfaction = (pref_time.apply(lambda r: matches_medium_time_preference(r["start_time"], r["preferred_time"]), axis=1).mean() * 100) if len(pref_time) else 0
    deviations = (pd.to_datetime(evaluation_df["date"]) - pd.to_datetime(evaluation_df["preferred_date"])).abs().dt.days
    average_date_deviation = float(deviations.mean()); maximum_date_deviation = int(deviations.max())
    scheduled_minutes = int(evaluation_df["appointment_duration"].sum())
else:
    preferred_doctor_satisfaction = preferred_date_satisfaction = preferred_time_satisfaction = 0
    average_date_deviation = maximum_date_deviation = scheduled_minutes = 0

constraint_violations = len(dynamic_baseline_violations_df)
dynamic_baseline_evaluation = pd.DataFrame({"Metric": [
    "Initial Appointment Requests", "Successful Cancellations", "New Appointment Requests", "Final Active Requests",
    "Scheduled Appointments", "Unscheduled Active Requests", "Successful Assignment Rate (%)", "New Request Scheduling Rate (%)",
    "Constraint Violations", "Preferred Doctor Satisfaction (%)", "Preferred Date Satisfaction (%)", "Preferred Time Satisfaction (%)",
    "Average Date Deviation (days)", "Maximum Date Deviation (days)", "Scheduled Minutes"
], "Value": [
    initial_requests, successful_cancellations, new_requests, final_active_requests, scheduled_requests, unscheduled_requests,
    round(successful_assignment_rate, 2), round(new_request_scheduling_rate, 2), constraint_violations,
    round(preferred_doctor_satisfaction, 2), round(preferred_date_satisfaction, 2), round(preferred_time_satisfaction, 2),
    round(average_date_deviation, 2), maximum_date_deviation, scheduled_minutes
]})
print("===== DYNAMIC SINGLE-AGENT BASELINE EVALUATION =====")
display(dynamic_baseline_evaluation)

===== DYNAMIC SINGLE-AGENT BASELINE EVALUATION =====


,Metric,Value
0,Initial Appointment Requests,3298.00
1,Successful Cancellations,100.00
2,New Appointment Requests,120.00
3,Final Active Requests,3318.00
4,Scheduled Appointments,3107.00
5,Unscheduled Active Requests,211.00
6,Successful Assignment Rate (%),93.64
7,New Request Scheduling Rate (%),78.33
8,Constraint Violations,0.00
9,Preferred Doctor Satisfaction (%),89.24


M9.13 Result

After applying the dynamic changes:

- Initial Appointment Requests: **3,298**
- Successful Cancellations: **100**
- New Appointment Requests: **120**
- Final Active Requests: **3,318**
- Scheduled Appointments: **3,107**
- Unscheduled Active Requests: **211**
- Successful Assignment Rate: **93.64%**
- New Request Scheduling Rate: **78.33%**
- Constraint Violations: **0**
- Preferred Doctor Satisfaction: **89.24%**
- Preferred Date Satisfaction: **84.55%**
- Preferred Time Satisfaction: **97.98%**
- Average Date Deviation: **0.31 days**
- Maximum Date Deviation: **7 days**
- Scheduled Minutes: **131,130**

These values are generated from the executed dynamic baseline and are used as the Single-Agent reference for the later MAS comparison.


M9.14 — Dynamic Event Performance Evaluation

This section evaluates how effectively the Single-Agent system handled the individual dynamic events.

Instead of evaluating only the final schedule, this analysis examines the system’s ability to process events successfully and recover appointments affected by disruptions.

The event-level evaluation is particularly important because a scheduling system may produce a valid final schedule while still struggling to respond effectively to individual disruptions.

In [37]:
# Part M9.14 — Dynamic Event Performance Evaluation
all_event_results_df = pd.concat([cancellation_results_df, unavailability_results_df, new_request_results_df, schedule_change_results_df], ignore_index=True)
total_events = len(all_event_results_df)
success_statuses = ["Successful", "Recovered", "Scheduled", "No Affected Appointment"]
successful_events = int(all_event_results_df["status"].isin(success_statuses).sum())
failed_events = total_events - successful_events
event_success_rate = successful_events / total_events * 100 if total_events else 0
total_affected_appointments = int(all_event_results_df["affected_appointments"].sum())
total_recovered_appointments = int(all_event_results_df["recovered_appointments"].sum())
recovery_rate = total_recovered_appointments / total_affected_appointments * 100 if total_affected_appointments else 0

event_type_summary = all_event_results_df.groupby("event_type").agg(
    Total_Events=("event_id", "count"), Affected_Appointments=("affected_appointments", "sum"), Recovered_Appointments=("recovered_appointments", "sum")
).reset_index()
success_by_type = all_event_results_df[all_event_results_df["status"].isin(success_statuses)].groupby("event_type").size().reset_index(name="Successful_Events")
event_type_summary = event_type_summary.merge(success_by_type, on="event_type", how="left")
event_type_summary["Successful_Events"] = event_type_summary["Successful_Events"].fillna(0)
event_type_summary["Event_Success_Rate (%)"] = (event_type_summary["Successful_Events"] / event_type_summary["Total_Events"] * 100).round(2)
recovery_denominator = event_type_summary["Affected_Appointments"].where(event_type_summary["Affected_Appointments"] != 0)
event_type_summary["Recovery_Rate (%)"] = (event_type_summary["Recovered_Appointments"] / recovery_denominator * 100).fillna(0).round(2)

dynamic_event_evaluation = pd.DataFrame({"Metric": ["Total Dynamic Events", "Successfully Processed Events", "Failed Events", "Event Success Rate (%)", "Affected Appointments", "Recovered Appointments", "Recovery Rate (%)"],
                                          "Value": [total_events, successful_events, failed_events, round(event_success_rate, 2), total_affected_appointments, total_recovered_appointments, round(recovery_rate, 2)]})
print("===== OVERALL DYNAMIC EVENT PERFORMANCE =====")
display(dynamic_event_evaluation)
print("===== EVENT TYPE PERFORMANCE =====")
display(event_type_summary)


===== OVERALL DYNAMIC EVENT PERFORMANCE =====


,Metric,Value
0,Total Dynamic Events,290.00
1,Successfully Processed Events,264.00
2,Failed Events,26.00
3,Event Success Rate (%),91.03
4,Affected Appointments,270.00
5,Recovered Appointments,170.00
6,Recovery Rate (%),62.96


===== EVENT TYPE PERFORMANCE =====


,event_type,Total_Events,Affected_Appointments,Recovered_Appointments,Successful_Events,Event_Success_Rate (%),Recovery_Rate (%)
0,Doctor Schedule Change,30,103,103,30,100.00,100.0
1,Doctor Unavailability,40,67,67,40,100.00,100.0
2,New Appointment Request,120,0,0,94,78.33,0.0
3,Patient Cancellation,100,100,0,100,100.00,0.0


M9.14 Result

The system processed **290 dynamic events**. The overall results were:

| Metric | Result |
|---|---:|
| Total Dynamic Events | 290 |
| Successfully Processed Events | 264 |
| Failed Events | 26 |
| Event Success Rate | 91.03% |
| Affected Appointments | 270 |
| Recovered Appointments | 170 |
| Overall Recovery Rate | 62.96% |

The 26 failed events correspond to new appointment requests that could not be scheduled under the available resource constraints. They are **not constraint violations**.

For disruption events affecting existing appointments, the recovery results are:
- Doctor Unavailability: **67 affected, 67 recovered (100% recovery)**
- Doctor Schedule Change: **103 affected, 103 recovered (100% recovery)**

Patient cancellations are removals rather than recovery events, and new appointment requests represent additional demand rather than recovery.


M9.15 — Dynamic Single-Agent Baseline Summary

This final section combines the overall scheduling evaluation and dynamic event evaluation into a single summary.

The combined results provide a concise view of the Single-Agent baseline performance before implementing and evaluating the Multi-Agent System.

These values will serve as the reference baseline for the later MAS experiment under the same dataset, constraints, and dynamic events.

In [38]:
# Part M9.15 — Dynamic Single-Agent Baseline Summary
dynamic_baseline_summary = pd.concat([dynamic_baseline_evaluation, dynamic_event_evaluation], ignore_index=True)
print("===== EXPERIMENT 2 — DYNAMIC SINGLE-AGENT BASELINE SUMMARY =====")
display(dynamic_baseline_summary)


===== EXPERIMENT 2 — DYNAMIC SINGLE-AGENT BASELINE SUMMARY =====


,Metric,Value
0,Initial Appointment Requests,3298.00
1,Successful Cancellations,100.00
2,New Appointment Requests,120.00
3,Final Active Requests,3318.00
4,Scheduled Appointments,3107.00
5,Unscheduled Active Requests,211.00
6,Successful Assignment Rate (%),93.64
7,New Request Scheduling Rate (%),78.33
8,Constraint Violations,0.00
9,Preferred Doctor Satisfaction (%),89.24


## Experiment 2 — Final Single-Agent Baseline Summary

Experiment 2 evaluates the Single-Agent scheduling system under a larger and dynamic hospital scheduling environment.

The experiment first creates the initial static schedule and validates it. It then introduces 290 dynamic events consisting of patient cancellations, doctor unavailability, new appointment requests, and doctor schedule changes.

The final performance values shown in the tables above are generated directly from the executed notebook. They include:

- final active appointment requests and scheduled appointments
- assignment rate and new-request scheduling rate
- constraint violations
- preference satisfaction
- date deviation
- dynamic event success rate
- affected and recovered appointments
- recovery rate and event-type performance

For doctor unavailability specifically, every generated event is anchored to an existing initial appointment, and the notebook verifies this condition before processing the events.

The resulting Experiment 2 baseline provides the dynamic Single-Agent reference that will later be compared with the Multi-Agent System under the same environment, constraints, event sequence, and evaluation metrics.


## Experimental Design Note — Single-Agent Reference for SA vs MAS

This notebook is the full Single-Agent baseline for the project comparison.

- **Environment 1:** Easy / Static — 20 doctors, 100 appointment requests, 6 scheduling days, 60-minute slots, and no dynamic events.
- **Environment 2:** Complex / Dynamic — 40 doctors, 2,200 patients, 3,298 initial appointment requests, 21 scheduling days, 30-minute base slots, multiple appointment durations, capacity constraints, and 290 dynamic events.
- **Fair comparison:** the Multi-Agent System should be evaluated using the same generated environment inputs, constraints, event sequence, random seeds, and evaluation metrics.
- **Purpose:** Experiment 1 establishes the static baseline, while Experiment 2 tests how the Single-Agent baseline performs when demand and scheduling conditions become dynamic.

The M4 live-demo notebook may use only the Easy / Static experiment for a short end-to-end demonstration, but this notebook retains both experiments as the full research baseline.
